# Hybrid Search 구현하기

1. **BM25 키워드 검색**과 **벡터 검색**의 원리와 차이를 설명할 수 있다
2. **RRF 기반 하이브리드 검색**을 구현할 수 있다
3. **Recall@5** 으로 검색 품질을 측정하고 세 방식을 비교할 수 있다

---
## 1. 환경 설정

필요한 패키지를 설치합니다. `uv`를 사용하면 훨씬 빠릅니다.

In [ ]:
# uv pip install (권장 — pip 대비 10~100x 빠름)
!uv pip install \
    langchain \
    langchain-community \
    langchain-anthropic \
    langchain-openai \
    langchain-chroma \
    langchain_huggingface \
    langfuse \
    chromadb \
    rank-bm25 \
    sentence-transformers \
    pymupdf \
    python-dotenv \
    matplotlib \
    pandas \
    --quiet

# 또는 pip를 사용하는 경우:
# !pip install langchain langchain-community langchain-elasticsearch \
#              langchain-anthropic \
#              langchain-openai langfuse chromadb rank-bm25 \
#              sentence-transformers python-dotenv matplotlib pandas -q

---
## 2. 라이브러리 로드 및 API 키 설정

`.env` 파일에서 API 키를 로드합니다. `.env` 파일이 없다면 아래 형식으로 생성하세요:

```
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com
```

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# .env 파일 로드 (프로젝트 루트 또는 현재 디렉토리)
load_dotenv()

# API 키 확인
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
LANGFUSE_PUBLIC_KEY = os.getenv("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.getenv("LANGFUSE_SECRET_KEY")
LANGFUSE_HOST = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

print("환경 변수 로드 결과:")
print(f"  OPENAI_API_KEY:    {'설정됨 ✓' if OPENAI_API_KEY else '없음 ✗ (.env 파일 확인 필요)'}")
print(f"  ANTHROPIC_API_KEY: {'설정됨 ✓' if ANTHROPIC_API_KEY else '없음 (선택사항)'}")
print(f"  LANGFUSE_PUBLIC_KEY: {'설정됨 ✓' if LANGFUSE_PUBLIC_KEY else '없음 (Langfuse 섹션 스킵 가능)'}")
print(f"  LANGFUSE_SECRET_KEY: {'설정됨 ✓' if LANGFUSE_SECRET_KEY else '없음 (Langfuse 섹션 스킵 가능)'}")

환경 변수 로드 결과:
  OPENAI_API_KEY:    설정됨 ✓
  ANTHROPIC_API_KEY: 설정됨 ✓
  LANGFUSE_PUBLIC_KEY: 설정됨 ✓
  LANGFUSE_SECRET_KEY: 설정됨 ✓


In [2]:
# 핵심 라이브러리 임포트
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
# Windows/Linux: matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료 ✓")

라이브러리 로드 완료 ✓


---
## 3. 샘플 문서 준비

한국어 AI/개발 기술 문서 20개를 직접 정의합니다.
각 문서는 `content`(본문)와 `metadata`(카테고리, ID)를 가집니다.

또한 Recall@10 측정을 위한 **테스트 질의와 정답 문서 ID**를 함께 정의합니다.

In [3]:
# 한국어 기술 문서 100+개 구성
# 핵심 30개 (수작업) + 카테고리별 유사 distractor (자동 생성)
# → Vector가 같은 카테고리 문서끼리 혼동하는 상황을 재현

# ── 핵심 문서 30개 ──
core_docs = [
    # LLM 모델
    {"id": "doc_01", "content": "GPT-4o는 OpenAI의 최신 멀티모달 모델로, 텍스트, 이미지, 오디오를 동시에 처리할 수 있습니다. API 가격은 input $5/1M tokens, output $15/1M tokens입니다.", "category": "llm"},
    {"id": "doc_02", "content": "Claude 3.5 Sonnet은 Anthropic이 개발한 대형 언어 모델입니다. 코딩, 분석, 창작 작업에서 뛰어난 성능을 보이며, 200K 토큰의 컨텍스트 창을 지원합니다.", "category": "llm"},
    {"id": "doc_03", "content": "Llama 3은 Meta가 공개한 오픈소스 LLM입니다. 8B, 70B, 405B 파라미터 버전이 있으며, 상업적 사용이 가능한 커뮤니티 라이선스를 제공합니다.", "category": "llm"},
    # 임베딩
    {"id": "doc_04", "content": "임베딩 모델은 텍스트를 고차원 벡터로 변환하는 모델입니다. text-embedding-3-small은 OpenAI의 경량 임베딩 모델로 1536 차원 벡터를 생성합니다.", "category": "embedding"},
    {"id": "doc_05", "content": "한국어 임베딩을 위해 jhgan/ko-sroberta-multitask 모델을 많이 사용합니다. 768 차원 벡터를 생성하며 KorSTS 벤치마크에서 우수한 성능을 보입니다.", "category": "embedding"},
    # RAG
    {"id": "doc_06", "content": "RAG(Retrieval-Augmented Generation)는 외부 문서를 검색하여 LLM의 답변 품질을 향상시키는 기법입니다. 환각(Hallucination)을 줄이고 최신 정보를 반영할 수 있습니다.", "category": "rag"},
    {"id": "doc_07", "content": "청킹(Chunking)은 긴 문서를 작은 조각으로 나누는 과정입니다. 청크 크기는 보통 256~512 토큰으로 설정하며, 오버랩은 20~50 토큰을 권장합니다.", "category": "rag"},
    {"id": "doc_08", "content": "리랭킹(Reranking)은 초기 검색 결과를 Cross-Encoder 모델로 재정렬하는 기법입니다. BM25나 벡터 검색의 상위 결과를 더 정교하게 필터링할 수 있습니다.", "category": "rag"},
    {"id": "doc_09", "content": "하이브리드 검색은 BM25 키워드 검색과 벡터 검색을 결합하는 방식입니다. RRF(Reciprocal Rank Fusion) 알고리즘으로 두 검색 결과를 통합하면 Recall이 크게 향상됩니다.", "category": "rag"},
    {"id": "doc_10", "content": "Self-RAG는 LLM이 검색 필요성을 스스로 판단하고, 검색 결과의 관련성과 답변 지지 여부를 평가하는 고급 RAG 기법입니다.", "category": "rag"},
    # 벡터DB
    {"id": "doc_11", "content": "Chroma는 오픈소스 벡터 데이터베이스로, Python에서 바로 사용할 수 있는 임베디드 모드를 지원합니다. 로컬 개발 및 프로토타이핑에 적합합니다.", "category": "vectordb"},
    {"id": "doc_12", "content": "Elasticsearch 8.x는 kNN 벡터 검색과 BM25 텍스트 검색을 네이티브로 지원합니다. rank.rrf 기능으로 두 검색을 쉽게 결합할 수 있습니다.", "category": "vectordb"},
    {"id": "doc_13", "content": "Pinecone은 완전 관리형 벡터 데이터베이스 서비스입니다. Starter 플랜에서 무료로 시작할 수 있으며, 수백만 벡터를 밀리초 단위로 검색합니다.", "category": "vectordb"},
    {"id": "doc_14", "content": "FAISS(Facebook AI Similarity Search)는 Meta가 개발한 고성능 벡터 유사도 검색 라이브러리입니다. HNSW, IVF 등 다양한 인덱스 알고리즘을 지원합니다.", "category": "vectordb"},
    # LangChain
    {"id": "doc_15", "content": "LangChain은 LLM 애플리케이션 개발을 위한 Python/JS 프레임워크입니다. Chain, Agent, Memory, Retriever 등의 추상화를 제공하여 복잡한 AI 파이프라인을 쉽게 구축할 수 있습니다.", "category": "framework"},
    {"id": "doc_16", "content": "LangChain EnsembleRetriever는 여러 Retriever를 가중치와 함께 결합합니다. 내부적으로 RRF 알고리즘을 사용하여 최종 순위를 계산합니다.", "category": "framework"},
    {"id": "doc_17", "content": "LangSmith는 LangChain의 LLM 애플리케이션 디버깅 및 모니터링 플랫폼입니다. 트레이스, 평가, 데이터셋 관리 기능을 제공합니다.", "category": "framework"},
    # 관측성/평가
    {"id": "doc_18", "content": "Langfuse는 LLM 애플리케이션을 위한 오픈소스 관측성 플랫폼입니다. 트레이스, 스팬, 스코어 기능으로 RAG 파이프라인의 품질을 모니터링할 수 있습니다.", "category": "observability"},
    {"id": "doc_19", "content": "Recall@K는 검색 품질 평가 지표입니다. 전체 관련 문서 중 상위 K개 결과 안에 포함된 관련 문서의 비율을 나타냅니다. RAG에서는 Recall@5를 주로 사용합니다.", "category": "evaluation"},
    {"id": "doc_20", "content": "RAGAS는 RAG 파이프라인 평가 프레임워크입니다. Faithfulness, Answer Relevancy, Context Precision, Context Recall 네 가지 지표로 RAG 품질을 자동 평가합니다.", "category": "evaluation"},
]

# ── 카테고리별 유사 distractor 자동 생성 ──
# 실전에서는 이렇게 비슷한 문서가 수백 개 존재합니다

synthetic_templates = {
    "llm": [
        ("GPT-4 Turbo", "OpenAI의 이전 세대 모델입니다. 128K 컨텍스트를 지원하며 API 가격은 input $10/1M tokens, output $30/1M tokens입니다."),
        ("GPT-4o mini", "OpenAI의 경량화 모델입니다. GPT-4o 대비 60% 저렴하며 API 가격은 input $0.15/1M tokens, output $0.60/1M tokens입니다."),
        ("GPT-3.5 Turbo", "OpenAI의 가성비 모델입니다. 16K 컨텍스트를 지원하며 API 가격은 input $0.5/1M tokens, output $1.5/1M tokens입니다."),
        ("Claude 3 Opus", "Anthropic의 최상위 모델입니다. 복잡한 추론에 강하며 API 가격은 input $15/1M tokens, output $75/1M tokens입니다."),
        ("Claude 3 Haiku", "Anthropic의 경량 모델입니다. 빠른 응답 속도가 장점이며 API 가격은 input $0.25/1M tokens, output $1.25/1M tokens입니다."),
        ("Gemini 1.5 Pro", "Google DeepMind의 멀티모달 모델입니다. 1M 토큰 초장문 컨텍스트를 지원하며 영상과 오디오 이해에 강합니다."),
        ("Gemini 1.5 Flash", "Google의 경량 멀티모달 모델입니다. 1M 컨텍스트를 지원하며 API 가격은 input $0.075/1M tokens입니다."),
        ("Mistral Large", "Mistral AI의 플래그십 모델입니다. 128K 컨텍스트를 지원하며, 유럽 기반 AI 기업으로 데이터 주권 이슈에 강합니다."),
        ("Qwen 2.5 72B", "Alibaba가 공개한 오픈소스 LLM입니다. 72B 파라미터 버전이 있으며, 중국어와 영어 이중 언어에 강합니다."),
        ("Phi-3 Medium", "Microsoft의 소형 언어 모델(SLM)입니다. 14B 파라미터로 경량이면서 GPT-3.5급 성능을 발휘합니다."),
    ],
    "embedding": [
        ("text-embedding-3-large", "OpenAI의 고성능 임베딩 모델입니다. 3072 차원 벡터를 생성하며 text-embedding-3-small 대비 정확도가 높습니다."),
        ("text-embedding-ada-002", "OpenAI의 이전 세대 임베딩 모델입니다. 1536 차원 벡터를 생성하며 현재는 text-embedding-3로 대체되었습니다."),
        ("BAAI/bge-m3", "BAAI의 다국어 임베딩 모델입니다. 1024 차원 벡터를 생성하며 Dense, Sparse, ColBERT 세 방식을 동시에 지원합니다."),
        ("BAAI/bge-large-en-v1.5", "BAAI의 영어 특화 임베딩 모델입니다. 1024 차원 벡터를 생성하며 MTEB 벤치마크에서 상위권입니다."),
        ("intfloat/multilingual-e5-large", "Microsoft의 다국어 임베딩 모델입니다. 1024 차원 벡터를 생성하며 100개 이상 언어를 지원합니다."),
        ("cohere-embed-v3", "Cohere의 상용 임베딩 모델입니다. 1024 차원 벡터를 생성하며 search_document와 search_query 입력 타입을 구분합니다."),
        ("voyage-large-2", "Voyage AI의 임베딩 모델입니다. 1536 차원 벡터를 생성하며 코드와 법률 문서에 특화되어 있습니다."),
        ("nomic-embed-text-v1.5", "Nomic AI의 오픈소스 임베딩 모델입니다. 768 차원 벡터를 생성하며 8192 토큰의 긴 문서를 처리할 수 있습니다."),
    ],
    "vectordb": [
        ("Weaviate", "오픈소스 벡터 데이터베이스로, GraphQL API와 하이브리드 검색을 기본 지원합니다. 스키마 기반 데이터 관리와 모듈식 벡터화를 제공합니다."),
        ("Milvus", "CNCF 소속 오픈소스 벡터 데이터베이스입니다. GPU 가속 검색을 지원하며 수십억 벡터를 처리할 수 있습니다."),
        ("Qdrant", "Rust로 개발된 오픈소스 벡터 데이터베이스입니다. 필터링과 페이로드 기반 검색을 지원하며 메모리 효율이 뛰어납니다."),
        ("pgvector", "PostgreSQL 확장 모듈로, 기존 RDB에 벡터 검색 기능을 추가합니다. ivfflat과 HNSW 인덱스를 지원합니다."),
        ("LanceDB", "서버리스 벡터 데이터베이스로, 로컬 파일 기반으로 동작합니다. Lance 컬럼형 포맷을 사용하여 디스크 기반 검색이 빠릅니다."),
        ("Vespa", "Yahoo가 개발한 대규모 검색 엔진입니다. 벡터 검색과 전통적 텍스트 검색, 구조화 데이터 검색을 모두 지원합니다."),
    ],
    "rag": [
        ("Corrective RAG(CRAG)", "검색된 문서가 질문과 관련 없으면 외부 웹 검색으로 보완하는 기법입니다. 검색 품질이 낮을 때 자동으로 대안을 찾습니다."),
        ("Adaptive RAG", "질문의 복잡도에 따라 검색 전략을 동적으로 변경하는 기법입니다. 단순 질문은 검색 없이, 복잡한 질문은 다단계 검색을 수행합니다."),
        ("Agentic RAG", "LLM 에이전트가 검색, 분석, 추론을 자율적으로 수행하는 고급 RAG 기법입니다. 도구 사용과 계획 수립이 가능합니다."),
        ("Multi-Query RAG", "원본 질문을 여러 관점에서 재구성하여 검색하는 기법입니다. 다양한 쿼리로 검색 범위를 넓혀 Recall을 향상시킵니다."),
        ("Parent Document Retriever", "작은 청크로 검색하되 원본 전체 문서를 반환하는 기법입니다. 검색 정확도와 컨텍스트 풍부함을 동시에 확보합니다."),
        ("Hypothetical Document Embeddings(HyDE)", "LLM이 가상 답변을 생성한 뒤 이를 임베딩하여 검색하는 기법입니다. 질문과 문서 간 임베딩 갭을 줄입니다."),
    ],
    "evaluation": [
        ("LLM-as-Judge", "LLM을 평가자로 활용하는 기법입니다. GPT-4 같은 모델이 응답의 정확성, 유용성, 안전성을 자동으로 점수화합니다."),
        ("MRR@K(Mean Reciprocal Rank)", "검색 품질 평가 지표입니다. 첫 번째 관련 문서의 순위 역수를 평균한 값으로, 1에 가까울수록 좋습니다."),
        ("NDCG@K", "검색 결과의 순위 품질을 평가하는 지표입니다. 관련성 높은 문서가 상위에 올수록 높은 점수를 받습니다."),
        ("Hit Rate@K", "검색 결과 상위 K개에 관련 문서가 하나라도 포함되어 있는지를 측정합니다. 이진(0/1) 평가 방식입니다."),
        ("Context Relevancy", "검색된 컨텍스트가 질문에 얼마나 관련 있는지를 LLM이 평가합니다. RAGAS 프레임워크의 핵심 지표 중 하나입니다."),
    ],

    "llm_extra": [
        ("Command R+", "Cohere의 RAG 특화 LLM입니다. 128K 컨텍스트를 지원하며 API 가격은 input $3/1M tokens, output $15/1M tokens입니다."),
        ("DBRX", "Databricks가 공개한 132B 파라미터 오픈소스 MoE 모델입니다. 32K 컨텍스트를 지원하며 추론 비용이 낮습니다."),
        ("Yi-34B", "01.AI가 공개한 오픈소스 LLM입니다. 34B 파라미터로 200K 컨텍스트를 지원하며 중국어와 영어에 강합니다."),
        ("DeepSeek V2", "DeepSeek이 개발한 236B MoE 모델입니다. 128K 컨텍스트를 지원하며 API 가격은 input $0.14/1M tokens으로 매우 저렴합니다."),
        ("Jamba 1.5", "AI21 Labs의 하이브리드 SSM-Transformer 모델입니다. 256K 컨텍스트를 지원하며 긴 문서 처리에 최적화되어 있습니다."),
        ("Nemotron-4 340B", "NVIDIA가 학습한 340B 파라미터 LLM입니다. 합성 데이터 생성에 특화되어 있으며 NVIDIA NIM으로 배포 가능합니다."),
        ("Arctic", "Snowflake이 공개한 480B MoE 모델입니다. 엔터프라이즈 SQL 생성과 코딩에 특화되어 있으며 Apache 2.0 라이선스입니다."),
    ],
    "embedding_extra": [
        ("Jina-embeddings-v3", "Jina AI의 다국어 임베딩 모델입니다. 1024 차원 벡터를 생성하며 8192 토큰의 긴 텍스트를 지원합니다."),
        ("mxbai-embed-large-v1", "Mixedbread AI의 임베딩 모델입니다. 1024 차원 벡터를 생성하며 MTEB 벤치마크 상위권입니다."),
        ("UAE-Large-V1", "WhereIsAI의 유니버설 임베딩 모델입니다. 1024 차원 벡터를 생성하며 AnglE 손실 함수로 학습되었습니다."),
        ("gte-large-en-v1.5", "Alibaba의 영어 임베딩 모델입니다. 1024 차원 벡터를 생성하며 MTEB 벤치마크에서 높은 성능을 보입니다."),
        ("snowflake-arctic-embed-l", "Snowflake의 임베딩 모델입니다. 1024 차원 벡터를 생성하며 검색 특화로 설계되었습니다."),
    ],
    "vectordb_extra": [
        ("Turbopuffer", "서버리스 벡터 데이터베이스로, 오브젝트 스토리지 기반으로 비용을 절감합니다. 콜드 스토리지 자동 전환을 지원합니다."),
        ("Marqo", "벡터 검색과 텍스트 검색을 통합한 엔드투엔드 검색 엔진입니다. 자동 임베딩 생성과 하이브리드 검색을 지원합니다."),
        ("Vald", "Yahoo Japan이 개발한 분산 벡터 검색 엔진입니다. Kubernetes 네이티브로 대규모 클러스터 운영에 적합합니다."),
        ("Redis Vector Search", "Redis Stack의 벡터 검색 모듈입니다. 인메모리 기반으로 초저지연 검색을 제공하며 기존 Redis 생태계와 통합됩니다."),
    ],
    "framework_extra": [
        ("Haystack", "deepset이 개발한 오픈소스 RAG 프레임워크입니다. 파이프라인 기반 아키텍처로 검색, 생성, 평가를 모듈식으로 구성합니다."),
        ("DSPy", "Stanford NLP 그룹이 개발한 프레임워크입니다. LLM 파이프라인을 프로그래밍 방식으로 최적화하며 프롬프트 자동 튜닝을 지원합니다."),
        ("CrewAI", "멀티 에이전트 오케스트레이션 프레임워크입니다. 역할 기반 에이전트 팀을 구성하여 복잡한 태스크를 협업으로 수행합니다."),
        ("AutoGen", "Microsoft가 개발한 멀티 에이전트 대화 프레임워크입니다. 에이전트 간 자동 대화를 통해 복잡한 문제를 해결합니다."),
        ("Semantic Kernel", "Microsoft의 AI 오케스트레이션 SDK입니다. C#과 Python을 지원하며 플러그인 기반으로 LLM 기능을 확장합니다."),
    ],
    "tools": [
        ("Unstructured.io", "비정형 문서(PDF, HTML, 이미지)를 구조화된 텍스트로 변환하는 도구입니다. RAG 파이프라인의 전처리 단계에 사용됩니다."),
        ("Docling", "IBM이 개발한 문서 파싱 도구입니다. PDF, DOCX, HTML 등을 마크다운으로 변환하며 테이블과 이미지 추출을 지원합니다."),
        ("Firecrawl", "웹 페이지를 LLM에 적합한 마크다운으로 크롤링하는 API 서비스입니다. JavaScript 렌더링과 동적 페이지 처리를 지원합니다."),
        ("Tavily", "AI 에이전트를 위한 검색 API입니다. 웹 검색 결과를 LLM에 적합한 형태로 요약하여 반환합니다."),
        ("Instructor", "LLM에서 구조화된 출력을 추출하는 라이브러리입니다. Pydantic 모델을 사용하여 타입 안전한 LLM 호출을 지원합니다."),
        ("Outlines", "LLM 출력을 정규식이나 JSON 스키마로 제한하는 라이브러리입니다. 구조화된 생성(Structured Generation)을 구현합니다."),
        ("vLLM", "고성능 LLM 추론 엔진입니다. PagedAttention 기술로 GPU 메모리를 효율적으로 관리하며 처리량을 극대화합니다."),
        ("Ollama", "로컬에서 LLM을 실행하는 도구입니다. Llama, Mistral 등 오픈소스 모델을 원클릭으로 설치하고 실행할 수 있습니다."),
    ],
    "cloud_ai": [
        ("Amazon Bedrock", "AWS의 관리형 AI 서비스입니다. Claude, Llama, Mistral 등 다양한 파운데이션 모델을 API로 제공하며 Knowledge Base 기반 RAG를 지원합니다."),
        ("Azure OpenAI Service", "Microsoft Azure에서 제공하는 OpenAI 모델 호스팅 서비스입니다. GPT-4, DALL-E, Whisper를 엔터프라이즈 보안 환경에서 사용할 수 있습니다."),
        ("Google Vertex AI", "Google Cloud의 MLOps 플랫폼입니다. Gemini 모델과 벡터 검색, 파인튜닝, 모델 평가를 통합 제공합니다."),
        ("Hugging Face Inference Endpoints", "Hugging Face의 모델 배포 서비스입니다. 오픈소스 모델을 전용 인프라에 배포하여 프로덕션 API로 제공합니다."),
        ("Together AI", "오픈소스 모델 호스팅 플랫폼입니다. Llama, Mistral 등을 API로 제공하며 파인튜닝과 추론 최적화를 지원합니다."),
        ("Groq", "LPU(Language Processing Unit) 기반 초고속 추론 서비스입니다. Llama, Mixtral 등 오픈소스 모델을 500 tokens/s 이상의 속도로 제공합니다."),
        ("Fireworks AI", "오픈소스 모델 추론 최적화 플랫폼입니다. 함수 호출과 JSON 모드를 지원하며 GPT-4급 모델을 저비용으로 제공합니다."),
        ("Replicate", "클라우드 기반 모델 실행 플랫폼입니다. Docker 컨테이너로 모델을 패키징하여 API로 배포하며 사용한 만큼만 과금합니다."),
        ("Anyscale", "Ray 기반의 분산 AI 플랫폼입니다. 대규모 LLM 파인튜닝과 서빙을 지원하며 OpenAI 호환 API를 제공합니다."),
        ("Modal", "서버리스 GPU 컴퓨팅 플랫폼입니다. Python 함수를 클라우드 GPU에서 실행하며 LLM 추론과 파인튜닝에 적합합니다."),
    ],
    "misc": [
        ("Few-shot 프롬프팅", "LLM에게 예시를 몇 개 보여준 뒤 유사한 패턴으로 답변하게 하는 기법입니다. 예시 수와 품질이 성능에 직접 영향을 미칩니다."),
        ("프롬프트 인젝션", "악의적 사용자가 LLM의 시스템 프롬프트를 우회하는 공격입니다. 입력 검증과 출력 필터링으로 방어할 수 있습니다."),
        ("컨텍스트 압축", "토큰 비용 최적화를 위해 불필요한 내용을 제거하고 핵심 정보만 LLM에 전달하는 기법입니다. Context Compression이라고도 합니다."),
        ("LlamaIndex", "RAG 파이프라인 구축에 특화된 프레임워크입니다. 다양한 데이터 소스(PDF, DB, API)에서 문서를 로드하고 인덱싱할 수 있습니다."),
        ("LangGraph", "LangChain 기반의 에이전트 오케스트레이션 프레임워크입니다. 상태 기반 그래프로 복잡한 멀티스텝 워크플로우를 구현합니다."),
        ("Guardrails AI", "LLM 출력의 품질과 안전성을 검증하는 프레임워크입니다. 스키마 검증, 유해 콘텐츠 필터링, 형식 보장 등의 가드레일을 제공합니다."),
    ],
}

# 자동 생성
synthetic_docs = []
doc_counter = 1
for category, items in synthetic_templates.items():
    for name, content in items:
        synthetic_docs.append({
            "id": f"syn_{doc_counter:03d}",
            "content": f"{name}은 {content}" if not content.startswith(name) else content,
            "category": category,
        })
        doc_counter += 1

# 통합
raw_docs = core_docs + synthetic_docs

# LangChain Document 객체로 변환
docs = [
    Document(
        page_content=d["content"],
        metadata={"id": d["id"], "category": d["category"]}
    )
    for d in raw_docs
]

print(f"총 문서 수: {len(docs)}개")
print(f"  - 핵심 문서 (수작업): {len(core_docs)}개")
print(f"  - Distractor (자동생성): {len(synthetic_docs)}개")
print(f"\n카테고리별 분포:")
from collections import Counter
cat_counts = Counter(d["category"] for d in raw_docs)
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:15s}: {cnt:3d}개")
print(f"\n→ LLM 모델만 13개, 임베딩 모델 10개, 벡터DB 10개")
print(f"   Vector는 같은 카테고리 문서를 구별하기 어려워짐!")
print(f"   K=3이면 전체의 {3/len(docs)*100:.1f}% — 훨씬 선별적")

총 문서 수: 100개
  - 핵심 문서 (수작업): 20개
  - Distractor (자동생성): 80개

카테고리별 분포:
  llm            :  13개
  rag            :  11개
  embedding      :  10개
  vectordb       :  10개
  cloud_ai       :  10개
  tools          :   8개
  evaluation     :   7개
  llm_extra      :   7개
  misc           :   6개
  embedding_extra:   5개
  framework_extra:   5개
  vectordb_extra :   4개
  framework      :   3개
  observability  :   1개

→ LLM 모델만 13개, 임베딩 모델 10개, 벡터DB 10개
   Vector는 같은 카테고리 문서를 구별하기 어려워짐!
   K=3이면 전체의 3.0% — 훨씬 선별적


In [4]:
# 테스트 질의 — 설계 원칙:
#   keyword  → 고유명사 제거, 숫자/스펙만으로 정답을 특정 (BM25 유리)
#   semantic → 정답 문서의 키워드를 완전히 피하고 동의어만 사용 (Vector 유리)
#   mixed    → relevant docs 2개, 하나는 BM25로만/하나는 Vector로만 도달 가능

test_queries = [
    # ── keyword (4개): 고유명사 없이 숫자/스펙으로만 특정 ──
    {
        "query": "input $5/1M tokens output $15/1M tokens 멀티모달 API 모델",
        "relevant_ids": ["doc_01"],
        "type": "keyword",
        # doc_01(GPT-4o: $5/$15) vs doc_d1(GPT-4 Turbo: $10/$30) vs doc_d2(Gemini)
        # BM25: "$5", "$15" 정확 매칭 → doc_01만 선택 ✓
        # Vector: 3개 LLM 가격 문서가 유사 벡터 → 혼동 ✗
    },
    {
        "query": "1536 차원 벡터 생성 경량 임베딩 모델",
        "relevant_ids": ["doc_04"],
        "type": "keyword",
        # doc_04(1536차원) vs doc_05(768차원) vs doc_d3(Dense+Sparse)
        # BM25: "1536" 정확 매칭 ✓
        # Vector: 임베딩 모델 3개가 같은 주제 → 혼동 ✗
    },
    {
        "query": "8B 70B 405B 세 가지 크기의 오픈소스 LLM 상업적 라이선스",
        "relevant_ids": ["doc_03"],
        "type": "keyword",
        # BM25: "8B", "70B", "405B" 정확 매칭 → doc_03만 ✓
        # Vector: 다른 LLM 문서(GPT-4o, Claude)도 유사 주제 ✗
    },
    {
        "query": "완전 관리형 서버리스 벡터 데이터베이스 무료 플랜 밀리초 검색",
        "relevant_ids": ["doc_13"],
        "type": "keyword",
        # doc_13(Pinecone) vs doc_11(Chroma) vs doc_14(FAISS) vs doc_d5(Weaviate)
        # BM25: "관리형", "서버리스", "밀리초" 조합으로 doc_13 특정 ✓
        # Vector: 벡터DB 4개 문서가 모두 유사 → 혼동 ✗
    },

    # ── semantic (4개): 정답 키워드를 의도적으로 회피 ──
    {
        "query": "AI가 사실이 아닌 내용을 지어내는 문제의 해결 방안",
        "relevant_ids": ["doc_06"],
        "type": "semantic",
        # doc_06: "환각(Hallucination)을 줄이고"
        # 쿼리에 "환각", "RAG", "Hallucination" 없음 → BM25 실패 ✗
        # "지어내는" ≈ "환각" 의미 매칭 → Vector 성공 ✓
    },
    {
        "query": "긴 글을 적절한 크기로 잘라서 색인 가능한 조각으로 만드는 전략",
        "relevant_ids": ["doc_07"],
        "type": "semantic",
        # doc_07: "청킹(Chunking)", "나누는 과정"
        # 쿼리에 "청킹", "Chunking" 없음 → BM25 실패 ✗
        # "잘라서 조각" ≈ "나누는" → Vector 성공 ✓
    },
    {
        "query": "AI 시스템의 동작을 관찰하고 품질을 수치로 추적하는 서비스",
        "relevant_ids": ["doc_18"],
        "type": "semantic",
        # doc_18: "관측성 플랫폼", "모니터링"
        # 쿼리에 "관측성", "Langfuse", "모니터링" 없음 → BM25 실패 ✗
        # "관찰/수치 추적" ≈ "관측성/스코어" → Vector 성공 ✓
    },
    {
        "query": "쓸데없는 내용을 걸러내서 LLM 호출 비용을 아끼는 방법",
        "relevant_ids": ["doc_d10"],
        "type": "semantic",
        # doc_d10: "컨텍스트 압축(Context Compression)", "비용을 절감"
        # 쿼리에 "압축", "Compression" 없음 → BM25 실패 ✗
        # "걸러내서 비용 아끼기" ≈ "불필요 제거, 비용 절감" → Vector 성공 ✓
    },

    # ── mixed (4개): 관련 문서 2개 — 하나는 BM25로, 하나는 Vector로만 도달 ──
    {
        "query": "Elasticsearch에서 의미 유사도와 단어 일치를 동시에 활용하는 검색",
        "relevant_ids": ["doc_12", "doc_09"],
        "type": "mixed",
        # "Elasticsearch" → BM25가 doc_12 찾음 ✓
        # "의미 유사도 동시 활용" ≈ "하이브리드 검색" → Vector가 doc_09 ✓
        # 단독: 각각 1개만, Hybrid: 둘 다 ✓
    },
    {
        "query": "RAGAS 프레임워크로 AI 응답의 신뢰도를 자동으로 점수 매기기",
        "relevant_ids": ["doc_20", "doc_d7"],
        "type": "mixed",
        # "RAGAS" → BM25가 doc_20 ✓
        # "자동으로 점수 매기기" ≈ "LLM-as-Judge" → Vector가 doc_d7 ✓
    },
    {
        "query": "LLM이 외부 지식을 참고하여 답변하되 검색 필요성을 스스로 판단하는 기법",
        "relevant_ids": ["doc_06", "doc_10"],
        "type": "mixed",
        # "스스로 판단" → BM25가 doc_10 ✓
        # "외부 지식 참고하여 답변" ≈ RAG → Vector가 doc_06 ✓
    },
    {
        "query": "Cross-Encoder 모델로 BM25 검색 결과의 순위를 다시 매기는 기법",
        "relevant_ids": ["doc_08"],
        "type": "mixed",
        # "Cross-Encoder", "BM25" → BM25가 doc_08 키워드 매칭 ✓
        # "순위를 다시 매기는" ≈ "재정렬" → Vector도 doc_08 매칭 ✓
        # 이 경우 둘 다 찾지만, BM25가 더 높은 순위로 배치
    },
]

print(f"테스트 질의 수: {len(test_queries)}개")
print(f"  - keyword:  {sum(1 for q in test_queries if q['type'] == 'keyword')}개")
print(f"  - semantic: {sum(1 for q in test_queries if q['type'] == 'semantic')}개")
print(f"  - mixed:    {sum(1 for q in test_queries if q['type'] == 'mixed')}개")

테스트 질의 수: 12개
  - keyword:  4개
  - semantic: 4개
  - mixed:    4개


---
## 4. BM25 검색 구현

**BM25(Best Match 25)** 는 단어 빈도(TF)와 역문서빈도(IDF)를 기반으로 한 전통적인 키워드 검색 알고리즘입니다.

LangChain의 `BM25Retriever`는 내부적으로 `rank-bm25` 라이브러리를 사용합니다.

In [5]:
from langchain_community.retrievers import BM25Retriever

# BM25 Retriever 생성
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 5  # 상위5개 반환

print("BM25 Retriever 생성 완료 ✓")

# 테스트: keyword 유형 쿼리
test_query = "Starter 플랜"
bm25_results = bm25_retriever.invoke(test_query)

print(f"\n쿼리: '{test_query}'")
print(f"BM25 검색 결과 (상위 5개):")
for i, doc in enumerate(bm25_results[:5], 1):
    print(f"  {i}. [{doc.metadata['id']}] {doc.page_content[:100]}...")

BM25 Retriever 생성 완료 ✓

쿼리: 'Starter 플랜'
BM25 검색 결과 (상위 5개):
  1. [doc_13] Pinecone은 완전 관리형 벡터 데이터베이스 서비스입니다. Starter 플랜에서 무료로 시작할 수 있으며, 수백만 벡터를 밀리초 단위로 검색합니다....
  2. [syn_080] Guardrails AI은 LLM 출력의 품질과 안전성을 검증하는 프레임워크입니다. 스키마 검증, 유해 콘텐츠 필터링, 형식 보장 등의 가드레일을 제공합니다....
  3. [syn_008] Mistral Large은 Mistral AI의 플래그십 모델입니다. 128K 컨텍스트를 지원하며, 유럽 기반 AI 기업으로 데이터 주권 이슈에 강합니다....
  4. [syn_009] Qwen 2.5 72B은 Alibaba가 공개한 오픈소스 LLM입니다. 72B 파라미터 버전이 있으며, 중국어와 영어 이중 언어에 강합니다....
  5. [syn_010] Phi-3 Medium은 Microsoft의 소형 언어 모델(SLM)입니다. 14B 파라미터로 경량이면서 GPT-3.5급 성능을 발휘합니다....


In [6]:
# BM25의 한계: 동의어/패러프레이즈를 사용하면 찾지 못한다
# doc_06 원문: "환각(Hallucination)을 줄이고 최신 정보를 반영"
# → 쿼리에서 공유 키워드를 모두 제거하면?

semantic_query = "AI가 헛소리를 지어내는 현상을 방지하는 접근법"

bm25_results = bm25_retriever.invoke(semantic_query)
bm25_top3_ids = [d.metadata['id'] for d in bm25_results[:3]]

print(f"쿼리: '{semantic_query}'")
print(f"정답: doc_06 (RAG로 환각을 줄이는 기법)")
print(f"      쿼리에 '환각', 'RAG', 'Hallucination' 단어가 없음!")
print()
print("[BM25 Top-3]")
for i, doc in enumerate(bm25_results[:3], 1):
    marker = "✓ 정답" if doc.metadata['id'] == 'doc_06' else "✗"
    print(f"  {i}. {marker} [{doc.metadata['id']}] {doc.page_content[:80]}...")

found = 'doc_06' in bm25_top3_ids
print(f"\n→ doc_06 Top-3 포함: {found}")
if not found:
    # 전체 순위에서 위치 확인
    all_ids = [d.metadata['id'] for d in bm25_results]
    if 'doc_06' in all_ids:
        rank = all_ids.index('doc_06') + 1
        print(f"  (전체 {len(all_ids)}개 중 {rank}위 — 상위권 밖)")
    else:
        print(f"  (검색 결과에 아예 없음)")
    print()
    print("★ BM25는 키워드가 다르면 관련 문서를 놓칩니다.")
    print("  이 문제를 해결하는 것이 벡터 검색과 하이브리드 검색입니다.")
    print("  → 섹션 5(벡터 검색)에서 같은 쿼리로 테스트해 봅니다.")

쿼리: 'AI가 헛소리를 지어내는 현상을 방지하는 접근법'
정답: doc_06 (RAG로 환각을 줄이는 기법)
      쿼리에 '환각', 'RAG', 'Hallucination' 단어가 없음!

[BM25 Top-3]
  1. ✗ [syn_080] Guardrails AI은 LLM 출력의 품질과 안전성을 검증하는 프레임워크입니다. 스키마 검증, 유해 콘텐츠 필터링, 형식 보장 등의 가드레일...
  2. ✗ [syn_017] voyage-large-2은 Voyage AI의 임베딩 모델입니다. 1536 차원 벡터를 생성하며 코드와 법률 문서에 특화되어 있습니다....
  3. ✗ [syn_007] Gemini 1.5 Flash은 Google의 경량 멀티모달 모델입니다. 1M 컨텍스트를 지원하며 API 가격은 input $0.075/1M t...

→ doc_06 Top-3 포함: False
  (검색 결과에 아예 없음)

★ BM25는 키워드가 다르면 관련 문서를 놓칩니다.
  이 문제를 해결하는 것이 벡터 검색과 하이브리드 검색입니다.
  → 섹션 5(벡터 검색)에서 같은 쿼리로 테스트해 봅니다.


---
## 5. 벡터 검색 구현

**[벡터 검색 프로세스]**
```mermaid
flowchart LR
    Q([사용자 쿼리]):::query --> E1(임베딩 모델):::model
    E1 --> QV[쿼리 벡터 공간]:::vector
    D([문서 DB]):::doc --> E2(임베딩 모델):::model
    E2 --> DV[문서 벡터 공간]:::vector
    QV --> SIM{유사도 계산}:::sim
    DV --> SIM
    SIM --> R([최종 검색 결과]):::result

    classDef query fill:#4285F4,color:#fff,stroke-width:2px
    classDef doc fill:#EA4335,color:#fff,stroke-width:2px
    classDef vector fill:#FBBC05,color:#000,stroke-width:2px
    classDef model fill:#9C27B0,color:#fff,stroke-width:2px
    classDef sim fill:#FF9800,color:#fff,stroke-width:2px
    classDef result fill:#34A853,color:#fff,stroke-width:2px,font-weight:bold
```

**Dense Retrieval**은 텍스트를 임베딩 벡터로 변환하고 코사인 유사도로 검색합니다.

- 임베딩 모델: `bge-m3` (한국어 포함 다국어 지원, 1024차원)
- 벡터 DB: `Chroma` (로컬 인메모리)

> **참고:** 최초 실행 시 모델 다운로드(약 300MB)가 필요합니다.

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

print("임베딩 모델 로드 중... (최초 실행 시 다운로드 필요)")

# 한국어 임베딩 모델 로드
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("임베딩 모델 로드 완료 ✓")

# ChromaDB에 문서 인덱싱
print("문서 벡터화 및 인덱싱 중...")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="hybrid_search_demo"
)

# 벡터 Retriever 생성
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 10}
)

print(f"인덱싱 완료: {len(docs)}개 문서 ✓")

임베딩 모델 로드 중... (최초 실행 시 다운로드 필요)


/var/folders/3q/k6nvdwh17tzc4mv5gyw77wx80000gn/T/ipykernel_27824/1619731931.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 61634.58it/s]


임베딩 모델 로드 완료 ✓
문서 벡터화 및 인덱싱 중...
인덱싱 완료: 100개 문서 ✓


In [8]:
# 벡터 검색 vs BM25: 각각 어떤 쿼리에서 유리한가?

print("=" * 70)
print("[테스트 1] 숫자/스펙으로 구별해야 하는 쿼리 (BM25 유리, Vector 불리)")
print("=" * 70)

# 핵심: 고유명사(Pinecone, GPT-4o)를 쓰지 않고 숫자/스펙만으로 질의
# → BM25는 "$5", "1536" 등 정확 매칭 가능
# → Vector는 비슷한 카테고리 문서끼리 혼동
spec_query = "input $5/1M tokens output $15/1M tokens"
# 정답: doc_01 (GPT-4o: $5/$15)
# distractor: doc_d1 (GPT-4 Turbo: $10/$30), doc_d2 (Gemini)

bm25_res = bm25_retriever.invoke(spec_query)
vector_res = vector_retriever.invoke(spec_query)

print(f"\n쿼리: '{spec_query}'")
print(f"정답: doc_01 (GPT-4o: input $5/1M, output $15/1M)")
print(f"함정: doc_d1 (GPT-4 Turbo: $10/$30), doc_d2 (Gemini) — 비슷하지만 다른 모델")
print()

print("[BM25 Top-3]  ← '$5', '$15' 정확 매칭")
for i, doc in enumerate(bm25_res[:3], 1):
    m = "✓" if doc.metadata['id'] == 'doc_01' else "✗"
    print(f"  {i}. {m} [{doc.metadata['id']}] {doc.page_content[:70]}...")

print()
print("[Vector Top-3]  ← LLM 가격 문서끼리 의미가 비슷해서 혼동")
for i, doc in enumerate(vector_res[:3], 1):
    m = "✓" if doc.metadata['id'] == 'doc_01' else "✗"
    print(f"  {i}. {m} [{doc.metadata['id']}] {doc.page_content[:70]}...")

bm25_rank = next((i for i, d in enumerate(bm25_res, 1) if d.metadata['id'] == 'doc_01'), '-')
vec_rank = next((i for i, d in enumerate(vector_res, 1) if d.metadata['id'] == 'doc_01'), '-')
print(f"\n→ doc_01 순위:  BM25={bm25_rank}위  vs  Vector={vec_rank}위")

print("\n" + "=" * 70)
print("[테스트 2] 동의어만 사용하는 쿼리 (BM25 불리, Vector 유리)")
print("=" * 70)

# 핵심: 정답 문서의 키워드를 의도적으로 회피
# → BM25는 공유 키워드가 없어서 실패
# → Vector는 의미적 유사성으로 성공
sem_query = "AI가 헛소리를 지어내는 현상을 방지하는 접근법"
# 정답: doc_06 ("환각(Hallucination)을 줄이고")
# 쿼리에 "환각", "RAG", "Hallucination", "정보" 없음

bm25_res2 = bm25_retriever.invoke(sem_query)
vector_res2 = vector_retriever.invoke(sem_query)

print(f"\n쿼리: '{sem_query}'")
print(f"정답: doc_06 ('환각을 줄이는 RAG 기법')")
print(f"      쿼리에 '환각', 'RAG', 'Hallucination' 단어 없음!")
print()

print("[BM25 Top-3]  ← 공유 키워드 없어서 엉뚱한 결과")
for i, doc in enumerate(bm25_res2[:3], 1):
    m = "✓" if doc.metadata['id'] == 'doc_06' else "✗"
    print(f"  {i}. {m} [{doc.metadata['id']}] {doc.page_content[:70]}...")

print()
print("[Vector Top-3]  ← '헛소리 방지' ≈ '환각 줄이기' 의미 매칭")
for i, doc in enumerate(vector_res2[:3], 1):
    m = "✓" if doc.metadata['id'] == 'doc_06' else "✗"
    print(f"  {i}. {m} [{doc.metadata['id']}] {doc.page_content[:70]}...")

bm25_rank2 = next((i for i, d in enumerate(bm25_res2, 1) if d.metadata['id'] == 'doc_06'), '-')
vec_rank2 = next((i for i, d in enumerate(vector_res2, 1) if d.metadata['id'] == 'doc_06'), '-')
print(f"\n→ doc_06 순위:  BM25={bm25_rank2}위  vs  Vector={vec_rank2}위")

print("\n" + "=" * 70)
print("[결론]")
print("=" * 70)
print("  테스트 1: 숫자/스펙 매칭 → BM25 승")
print("  테스트 2: 동의어/패러프레이즈 → Vector 승")
print("  → 둘 다 잘하는 방법은? Hybrid Search (섹션 6)")

[테스트 1] 숫자/스펙으로 구별해야 하는 쿼리 (BM25 유리, Vector 불리)

쿼리: 'input $5/1M tokens output $15/1M tokens'
정답: doc_01 (GPT-4o: input $5/1M, output $15/1M)
함정: doc_d1 (GPT-4 Turbo: $10/$30), doc_d2 (Gemini) — 비슷하지만 다른 모델

[BM25 Top-3]  ← '$5', '$15' 정확 매칭
  1. ✓ [doc_01] GPT-4o는 OpenAI의 최신 멀티모달 모델로, 텍스트, 이미지, 오디오를 동시에 처리할 수 있습니다. API 가격은 in...
  2. ✗ [syn_036] Command R+은 Cohere의 RAG 특화 LLM입니다. 128K 컨텍스트를 지원하며 API 가격은 input $3/1M...
  3. ✗ [syn_004] Claude 3 Opus은 Anthropic의 최상위 모델입니다. 복잡한 추론에 강하며 API 가격은 input $15/1M ...

[Vector Top-3]  ← LLM 가격 문서끼리 의미가 비슷해서 혼동
  1. ✗ [syn_003] GPT-3.5 Turbo은 OpenAI의 가성비 모델입니다. 16K 컨텍스트를 지원하며 API 가격은 input $0.5/1M...
  2. ✗ [syn_036] Command R+은 Cohere의 RAG 특화 LLM입니다. 128K 컨텍스트를 지원하며 API 가격은 input $3/1M...
  3. ✗ [syn_007] Gemini 1.5 Flash은 Google의 경량 멀티모달 모델입니다. 1M 컨텍스트를 지원하며 API 가격은 input $...

→ doc_01 순위:  BM25=1위  vs  Vector=8위

[테스트 2] 동의어만 사용하는 쿼리 (BM25 불리, Vector 유리)

쿼리: 'AI가 헛소리를 지어내는 현상을 방지하는 접근법'
정답: doc_06 ('환각을 줄이는 RAG 기법')
      쿼리에 '환각', 'RAG', '

---
## 6. RRF 하이브리드 결합

**[하이브리드 검색 아키텍처]**
```mermaid
flowchart LR
    Q([질의]):::query --> BM25[BM25 Retrieval]:::bm25
    Q --> VEC[Vector Retrieval]:::vec
    BM25 --> R1(Ranked List 1):::dlist
    VEC --> R2(Ranked List 2):::dlist
    R1 --> RRF{RRF Fusion}:::rrf
    R2 --> RRF
    RRF --> F([최종 검색 결과]):::final

    classDef query fill:#4285F4,color:#fff,stroke-width:2px
    classDef bm25 fill:#EA4335,color:#fff,stroke-width:2px
    classDef vec fill:#FBBC05,color:#000,stroke-width:2px
    classDef dlist fill:#f5f5f5,color:#333,stroke-width:2px
    classDef rrf fill:#0f3460,color:#fff,font-weight:bold,stroke-width:2px
    classDef final fill:#34A853,color:#fff,stroke-width:2px,font-weight:bold
```

**Reciprocal Rank Fusion (RRF)** 공식:

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$


In [9]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from typing import List, Optional

class PandasHybridRRFRetriever(BaseRetriever):
    """BM25와 Vector 검색을 RRF로 결합하는 커스텀 리트리버"""
    
    retrievers: List[BaseRetriever]
    weights: Optional[List[float]] = None
    k: int = 60

    class Config:
        """Pydantic에서 임의의 타입 허용을 위한 설정"""
        arbitrary_types_allowed = True

    @classmethod
    def compute_weighted_rrf(
        cls, 
        rankings: List[List[Document]], 
        weights: Optional[List[float]] = None, 
        k: int = 60
    ) -> List[Document]:
        """클래스 레벨에서 호출 가능한 RRF 계산 로직"""
        if weights is None:
            weights = [1.0] * len(rankings)
            
        rrf_scores = {}
        for i, docs in enumerate(rankings):
            current_weight = weights[i]
            for rank, doc in enumerate(docs, start=1):
                # ID가 없다면 page_content를 고유 키로 사용
                doc_id = doc.metadata.get('id') or doc.page_content 
                score = current_weight * (1.0 / (k + rank))
                
                if doc_id not in rrf_scores:
                    rrf_scores[doc_id] = {"score": score, "doc": doc}
                else:
                    rrf_scores[doc_id]["score"] += score

        sorted_results = sorted(
            rrf_scores.values(), 
            key=lambda x: x["score"], 
            reverse=True
        )
        return [item["doc"] for item in sorted_results]

    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun
    ) -> List[Document]:
        # 1. 모든 리트리버 실행
        rankings = [r.invoke(query) for r in self.retrievers]
        
        # 2. 클래스 메서드 호출 (cls 대신 인스턴스에서 self. 혹은 클래스명으로 호출 가능)
        return self.compute_weighted_rrf(
            rankings=rankings, 
            weights=self.weights, 
            k=self.k
        )

# --- 사용 예시 ---
ensemble_retriever = PandasHybridRRFRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
    k=60
)

# 하이브리드 검색 테스트
print("\n" + "=" * 60)
print("하이브리드 검색 결과")
print("=" * 60)

for test in test_queries[:3]:
    results = ensemble_retriever.invoke(test["query"])
    result_ids = [d.metadata['id'] for d in results[:10]]
    
    hit = any(rid in result_ids for rid in test['relevant_ids'])
    print(f"\n[{test['type']}] {test['query']}")
    print(f"  정답 ID: {test['relevant_ids']}")
    print(f"  검색 결과 상위 5: {result_ids[:5]}")
    print(f"  정답 포함: {'✓' if hit else '✗'}")



하이브리드 검색 결과

[keyword] input $5/1M tokens output $15/1M tokens 멀티모달 API 모델
  정답 ID: ['doc_01']
  검색 결과 상위 5: ['doc_01', 'syn_007', 'syn_004', 'syn_003', 'syn_036']
  정답 포함: ✓

[keyword] 1536 차원 벡터 생성 경량 임베딩 모델
  정답 ID: ['doc_04']
  검색 결과 상위 5: ['doc_04', 'syn_017', 'syn_012', 'syn_014', 'syn_067']
  정답 포함: ✓

[keyword] 8B 70B 405B 세 가지 크기의 오픈소스 LLM 상업적 라이선스
  정답 ID: ['doc_03']
  검색 결과 상위 5: ['doc_03', 'syn_062', 'syn_013', 'syn_009', 'doc_20']
  정답 포함: ✓


---
## 7. Recall@K 측정 함수 구현

**Recall@K** = (상위 K개 결과 중 관련 문서 수) / (전체 관련 문서 수)

예: 관련 문서가 2개이고, 상위 10개 결과 중 1개가 관련 문서라면 Recall@10 = 0.5

In [10]:
def compute_recall_at_k(retriever, queries: list, k: int = 10) -> dict:
    """
    Recall@K + MRR@K 측정.
    
    Recall@K = (상위 K개 중 관련 문서 수) / (전체 관련 문서 수)
    MRR@K    = 1 / (첫 번째 관련 문서의 순위)  ← 순위 품질 측정
    """
    per_query_results = []
    
    for q in queries:
        results = retriever.invoke(q["query"])
        result_ids = [doc.metadata['id'] for doc in results[:k]]
        
        relevant_found = sum(1 for rid in q["relevant_ids"] if rid in result_ids)
        recall = relevant_found / len(q["relevant_ids"]) if q["relevant_ids"] else 0
        
        # MRR: 첫 번째 관련 문서의 역순위
        mrr = 0.0
        for rank, rid in enumerate(result_ids, 1):
            if rid in q["relevant_ids"]:
                mrr = 1.0 / rank
                break
        
        per_query_results.append({
            "query": q["query"],
            "type": q["type"],
            "recall": recall,
            "mrr": mrr,
            "relevant_found": relevant_found,
            "total_relevant": len(q["relevant_ids"]),
            "first_relevant_rank": next((i+1 for i, rid in enumerate(result_ids) if rid in q["relevant_ids"]), None),
        })
    
    overall_recall = sum(r["recall"] for r in per_query_results) / len(per_query_results)
    overall_mrr = sum(r["mrr"] for r in per_query_results) / len(per_query_results)
    
    by_type = {}
    for q_type in set(r["type"] for r in per_query_results):
        type_results = [r for r in per_query_results if r["type"] == q_type]
        by_type[q_type] = {
            "recall": sum(r["recall"] for r in type_results) / len(type_results),
            "mrr": sum(r["mrr"] for r in type_results) / len(type_results),
        }
    
    return {
        "overall_recall": overall_recall,
        "overall_mrr": overall_mrr,
        "by_type": by_type,
        "per_query": per_query_results,
    }

print("compute_recall_at_k() 정의 완료 (Recall@K + MRR@K)")

compute_recall_at_k() 정의 완료 (Recall@K + MRR@K)


In [11]:
import pandas as pd

# 3개 리트리버 비교
retrievers_to_compare = {
    'BM25': bm25_retriever,
    'Vector': vector_retriever,
    'Hybrid (RRF)': ensemble_retriever,
}

# K=3 기준 쿼리 유형별 Recall + MRR
print('[쿼리 유형별 Recall@3 + MRR@3]')
print('=' * 60)

type_rows = []
for ret_name, ret in retrievers_to_compare.items():
    result = compute_recall_at_k(ret, test_queries, k=3)
    for qtype, metrics in result['by_type'].items():
        type_rows.append({
            'Retriever': ret_name,
            'type': qtype,
            'Recall@3': round(metrics['recall'], 3),
            'MRR@3': round(metrics['mrr'], 3),
        })

df_type = pd.DataFrame(type_rows)
for qtype in ['keyword', 'semantic', 'mixed']:
    print(f'\n  [{qtype}]')
    sub = df_type[df_type['type'] == qtype][['Retriever', 'Recall@3', 'MRR@3']].set_index('Retriever')
    sub = sub.reindex(['BM25', 'Vector', 'Hybrid (RRF)'])
    display(sub)

# 쿼리별 순위 상세
print('\n[쿼리별 정답 첫 등장 순위]')
print(f"{'쿼리':<40} {'유형':<9} {'BM25':>5} {'Vector':>7} {'Hybrid':>7}")
print("-" * 72)
for q in test_queries:
    ranks = {}
    for ret_name, ret in retrievers_to_compare.items():
        r = compute_recall_at_k(ret, [q], k=10)
        rank = r['per_query'][0]['first_relevant_rank']
        ranks[ret_name] = f"{rank}위" if rank else "-"
    print(f"{q['query'][:38]:<40} {q['type']:<9} {ranks['BM25']:>5} {ranks['Vector']:>7} {ranks['Hybrid (RRF)']:>7}")

print()
print("=" * 60)
print("[인사이트] 왜 인라인 문서에서는 차이가 작은가?")
print("=" * 60)
print()
print("1. 문서 100개 = 코퍼스가 작다")
print("   → K=3이면 전체의 3%, 대부분 찾아짐")
print()
print("2. 문서가 짧고 주제가 뚜렷하다")
print("   → bge-m3 같은 최신 임베딩 모델은 짧은 문서를 쉽게 구별")
print("   → 숫자($5, 1536)까지 벡터 공간에서 잘 구분")
print()
print("3. 그래서 Vector만으로 거의 충분하다")
print("   → 이건 실패가 아니라 '작은 코퍼스의 특성'")
print()
print("★ 그러나 실전은 다릅니다:")
print("   → 수백~수천 청크, 비슷한 내용이 반복되는 문서")
print("   → 법률: 제23조~제28조가 모두 '해고' 관련 → Vector가 혼동")
print("   → 금융: 재무 테이블이 수십 개 → 숫자 구별 필요")
print("   → 섹션 8에서 실전 PDF로 확인합니다!")

[쿼리 유형별 Recall@3 + MRR@3]

  [keyword]


,Recall@3,MRR@3
Retriever,,
BM25,1.0,1.000
Vector,1.0,0.875
Hybrid (RRF),1.0,1.000



  [semantic]


,Recall@3,MRR@3
Retriever,,
BM25,0.5,0.375
Vector,0.5,0.375
Hybrid (RRF),0.5,0.500



  [mixed]


,Recall@3,MRR@3
Retriever,,
BM25,0.500,0.500
Vector,0.875,1.000
Hybrid (RRF),0.625,0.625



[쿼리별 정답 첫 등장 순위]
쿼리                                       유형         BM25  Vector  Hybrid
------------------------------------------------------------------------
input $5/1M tokens output $15/1M token   keyword      1위      2위      1위
1536 차원 벡터 생성 경량 임베딩 모델                  keyword      1위      1위      1위
8B 70B 405B 세 가지 크기의 오픈소스 LLM 상업적 라이선스   keyword      1위      1위      1위
완전 관리형 서버리스 벡터 데이터베이스 무료 플랜 밀리초 검색       keyword      1위      1위      1위
AI가 사실이 아닌 내용을 지어내는 문제의 해결 방안            semantic      -      2위      4위
긴 글을 적절한 크기로 잘라서 색인 가능한 조각으로 만드는 전략      semantic     1위      1위      1위
AI 시스템의 동작을 관찰하고 품질을 수치로 추적하는 서비스        semantic     2위      5위      1위
쓸데없는 내용을 걸러내서 LLM 호출 비용을 아끼는 방법          semantic      -       -       -
Elasticsearch에서 의미 유사도와 단어 일치를 동시에 활용하   mixed         -      1위      2위
RAGAS 프레임워크로 AI 응답의 신뢰도를 자동으로 점수 매기기     mixed         -      1위      4위
LLM이 외부 지식을 참고하여 답변하되 검색 필요성을 스스로 판단하는   mixed        1위      1위      1위
Cross-Encoder 모델로 BM25 검색 결과의 순위를

---
## 8. 실전 데이터로 Recall@K 실험

지금까지 인라인 문서 30개로 기본 사용법을 익혔습니다.
여기서부터는 **실제 PDF 문서**(`data/` 폴더)를 활용해 프로덕션 수준의 검색 성능 실험을 수행합니다.

### 데이터셋 구성

| 파일 | 내용 | 언어 | 도메인 | BM25 강점 |
|------|------|------|--------|----------|
| `20240531_company_652771000.pdf` | 삼성전기(009150) 증권사 리포트 | 한국어 | 금융 | 종목코드, 목표주가, EPS 등 정확한 수치 |
| `labor_law.pdf` | 근로기준법 전문 | 한국어 | 법률 | 조문 번호(제26조), 기간(30일), 비율(100분의 40) |
| `transformer.pdf` | Attention Is All You Need (2017) | 영어 | AI | BLEU 28.4, d_model=512, 수식 기호 |

### 왜 이 데이터에서 Hybrid가 빛나는가?

금융·법률 문서는 **정확한 숫자/코드**가 핵심이라 BM25가 필수입니다.
하지만 동시에 "해고당했을 때 구제 방법"처럼 **일상어로 검색**하는 경우도 많아서 Vector도 필요합니다.

→ **실전 문서일수록 Hybrid가 유리** (인라인 샘플과 다른 패턴)

### 실험 목표

- 숫자/코드 쿼리 vs 의미 쿼리에서 BM25와 Vector의 성능 격차 확인
- 이종 문서(heterogeneous corpus) 환경에서 Hybrid의 안정성 검증
- RRF 가중치 최적화를 통한 성능 극대화

In [12]:
# --- 8-1. PDF 로드 ---
import fitz  # PyMuPDF
import os

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), '..', 'data')
# 절대 경로로 보정
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'data'))
print(f'데이터 디렉토리: {DATA_DIR}')

def load_pdf(path: str) -> str:
    """PDF를 텍스트로 변환 (PyMuPDF)"""
    doc = fitz.open(path)
    text = ''
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

pdf_files = {
    'samsung_electro': '20240531_company_652771000.pdf',  # 삼성전기 증권사 리포트
    'labor_law':       'labor_law.pdf',                   # 근로기준법 상세
    'transformer':     'transformer.pdf',                 # Attention Is All You Need
}

raw_texts = {}
for key, filename in pdf_files.items():
    path = os.path.join(DATA_DIR, filename)
    raw_texts[key] = load_pdf(path)
    print(f'{key:20s} | {len(raw_texts[key]):>8,} chars | {filename}')

print(f'\n총 {sum(len(v) for v in raw_texts.values()):,} chars 로드 완료')

데이터 디렉토리: /Users/pdstudio/Work/youtubu-pdstudio/data
samsung_electro      |    7,247 chars | 20240531_company_652771000.pdf
labor_law            |   39,524 chars | labor_law.pdf
transformer          |   39,498 chars | transformer.pdf

총 86,269 chars 로드 완료


In [13]:
# --- 8-2. 청킹 (Chunking) ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ',],
)

pdf_docs = []
for key, text in raw_texts.items():
    if 'samsung' in key:
        domain = 'finance'
    elif 'labor' in key:
        domain = 'law'
    else:
        domain = 'ai'
    chunks = splitter.split_text(text)
    for i, chunk in enumerate(chunks):
        pdf_docs.append(Document(
            page_content=chunk,
            metadata={
                'id': f'{key}_chunk_{i:03d}',
                'source': key,
                'domain': domain,
                'chunk_index': i,
            }
        ))

print(f'청크 크기: {CHUNK_SIZE} / 오버랩: {CHUNK_OVERLAP}')
print(f'총 청크 수: {len(pdf_docs)}')
print()
for key in raw_texts:
    n = sum(1 for d in pdf_docs if d.metadata['source'] == key)
    print(f'  {key:20s} → {n:>4d} chunks')

청크 크기: 500 / 오버랩: 100
총 청크 수: 220

  samsung_electro      →   19 chunks
  labor_law            →   99 chunks
  transformer          →  102 chunks


In [14]:
# --- 8-3. Ground Truth 생성: 이중 전략 ---
#
# keyword 쿼리 → strict 키워드 매칭 (정확한 숫자/코드 포함 청크만)
#   → "제26조"와 "30일"이 모두 있는 청크만 정답
#   → BM25가 유리하고, Vector는 비슷한 주제 청크와 혼동
#
# semantic 쿼리 → LLM-as-Judge (의미적 판단)
#   → "쫓겨났을 때" ≈ "해고" 의미 매칭 필요
#   → Vector가 유리하고, BM25는 키워드 불일치로 실패

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

def strict_match(must_contain: list, source_filter: str = None):
    """keyword 쿼리용: 모든 키워드가 정확히 포함된 청크만 정답"""
    def fn(doc):
        text = doc.page_content
        if source_filter and doc.metadata.get('source') != source_filter:
            return False
        return all(kw in text for kw in must_contain)
    return fn

eval_queries_raw = [
    # ── 금융: keyword (종목코드, 가격, EPS 등 정확 수치) ──
    {
        'query': '삼성전기 목표주가 200,000원 투자의견 BUY',
        'domain': 'finance', 'type': 'keyword',
        'ground_truth_fn': strict_match(['200,000', 'BUY'], 'samsung_electro'),
        # 정답: "목표주가 200,000원"과 "BUY"가 모두 있는 청크만
        # Vector: "삼성전기 투자" 관련 청크를 여러 개 반환하지만 정확한 수치 구별 못 함
    },
    {
        'query': '삼성전기 EPS 8,999원 PER 17.4배 2024F',
        'domain': 'finance', 'type': 'keyword',
        'ground_truth_fn': strict_match(['8,999', 'PER'], 'samsung_electro'),
        # BM25: "8,999", "17.4" 정확 매칭
        # Vector: 재무 데이터 테이블 청크들이 비슷한 벡터 공간에 뭉침
    },

    # ── 금융: semantic (일상어, 수치 없이 질의) ──
    {
        'query': '이 회사 주식을 지금 사도 되는지, 앞으로 실적이 좋아질 근거',
        'domain': 'finance', 'type': 'semantic',
        'ground_truth_fn': None,  # LLM-as-Judge 사용
        # 쿼리에 "BUY", "목표주가", "컴포넌트" 없음 → BM25 실패
    },
    {
        'query': '전자부품 각 사업 부문의 실적이 어떻게 변하고 있는지',
        'domain': 'finance', 'type': 'semantic',
        'ground_truth_fn': None,
    },

    # ── 법률: keyword (조문 번호, 기간, 비율) ──
    {
        'query': '제26조 해고의 예고 30일 전 통상임금 30일분',
        'domain': 'law', 'type': 'keyword',
        'ground_truth_fn': strict_match(['제26조', '30일']),
        # BM25: "제26조", "30일" 정확 매칭 → 딱 1~2개 청크
        # Vector: 해고 관련 여러 조항(제23조~28조)이 모두 비슷한 벡터
    },
    {
        'query': '제37조 지연이자 연 100분의 40 미지급 임금',
        'domain': 'law', 'type': 'keyword',
        'ground_truth_fn': strict_match(['제37조', '100분의 40']),
        # BM25: "제37조", "100분의 40" 정확 매칭
        # Vector: 임금 관련 조항 여러 개와 혼동
    },

    # ── 법률: semantic (일상어) ──
    {
        'query': '직장에서 부당하게 쫓겨났을 때 도움받을 수 있는 곳과 절차',
        'domain': 'law', 'type': 'semantic',
        'ground_truth_fn': None,
        # 쿼리에 "해고", "노동위원회", "구제" 없음 → BM25 실패
    },
    {
        'query': '회사가 월급을 제때 안 줄 때 받을 수 있는 벌칙성 이자',
        'domain': 'law', 'type': 'semantic',
        'ground_truth_fn': None,
        # 쿼리에 "임금", "지연이자", "제37조" 없음 → BM25 실패
    },

    # ── AI: keyword (수치, 하이퍼파라미터) ──
    {
        'query': 'BLEU score 28.4 EN-DE WMT 2014 translation',
        'domain': 'ai', 'type': 'keyword',
        'ground_truth_fn': strict_match(['28.4', 'EN-DE'], 'transformer'),
        # BM25: "28.4", "EN-DE" 정확 매칭
        # Vector: NLP 벤치마크 결과 청크들이 비슷
    },
    {
        'query': 'd_model 512 d_ff 2048 h 8 heads Transformer',
        'domain': 'ai', 'type': 'keyword',
        'ground_truth_fn': strict_match(['512', '2048'], 'transformer'),
    },

    # ── AI: semantic (패러프레이즈) ──
    {
        'query': '순서 정보가 없는 구조에서 단어 위치를 알려주기 위해 추가하는 신호',
        'domain': 'ai', 'type': 'semantic',
        'ground_truth_fn': None,
        # 정답: positional encoding 관련 청크
        # 쿼리에 "positional encoding" 없음 → BM25 실패
    },
    {
        'query': '순환 구조의 순차 처리 병목을 제거하고 모든 위치를 동시에 처리하는 방법',
        'domain': 'ai', 'type': 'semantic',
        'ground_truth_fn': None,
        # 정답: self-attention / parallelization 관련 청크
        # 쿼리에 "self-attention", "RNN" 없음 → BM25 실패
    },
]

print(f'평가 쿼리: {len(eval_queries_raw)}개')
print(f'  keyword:  {sum(1 for q in eval_queries_raw if q["type"] == "keyword")}개 (strict 매칭 → BM25 유리)')
print(f'  semantic: {sum(1 for q in eval_queries_raw if q["type"] == "semantic")}개 (LLM-as-Judge → Vector 유리)')

평가 쿼리: 12개
  keyword:  6개 (strict 매칭 → BM25 유리)
  semantic: 6개 (LLM-as-Judge → Vector 유리)


In [20]:
# --- 8-4. Ground Truth 라벨링 (이중 전략) ---

JUDGE_PROMPT = '''당신은 검색 품질 평가 전문가입니다.
아래 질문에 대해 주어진 텍스트 청크가 답변에 직접적으로 도움이 되는지 판단하세요.

## 질문
{query}

## 텍스트 청크
{chunk}

## 판정 기준
- RELEVANT: 이 청크가 질문에 대한 직접적인 답변 정보를 포함
- NOT_RELEVANT: 주제가 비슷하더라도 질문의 답변에 직접 도움이 안 됨

한 단어로만 답하세요: RELEVANT 또는 NOT_RELEVANT'''

def label_with_llm(query, chunks, domain, vectorstore=None, top_k=20):
    """
    2단계 라벨링으로 LLM 비용 90% 절감:
      1단계: 벡터 검색으로 Top-K 후보 선별 (무료)
      2단계: LLM이 후보만 판정 (유료, but K개만)
    """
    domain_map = {
        'finance': ['samsung_electro'],
        'law': ['labor_law'],
        'ai': ['transformer'],
    }
    valid_sources = domain_map.get(domain, [])
    
    # 1단계: 벡터 검색 pre-filter (비용 0)
    if vectorstore is not None:
        # 벡터 검색으로 상위 후보만 추출
        vec_results = vectorstore.similarity_search(query, k=top_k * 3)
        # 도메인 필터링
        candidates = [d for d in vec_results if d.metadata.get('source') in valid_sources][:top_k]
    else:
        # vectorstore 없으면 도메인 필터링만
        candidates = [d for d in chunks if d.metadata.get('source') in valid_sources]
    
    # 2단계: LLM 판정 (후보만)
    relevant_ids = []
    for doc in candidates:
        if len(doc.page_content.strip()) < 50:
            continue
        resp = llm.invoke(JUDGE_PROMPT.format(query=query, chunk=doc.page_content[:500])).content.strip()
        if 'RELEVANT' in resp and 'NOT_RELEVANT' not in resp:
            relevant_ids.append(doc.metadata['id'])
    return relevant_ids

# 라벨링 실행
eval_queries = []
print('Ground Truth 라벨링 진행 중...')
print('=' * 70)

for i, q in enumerate(eval_queries_raw):
    print(f'  [{i+1:2d}/{len(eval_queries_raw)}] [{q["type"]:8s}] {q["query"][:45]}...', end=' ')
    
    if q['type'] == 'keyword' and q.get('ground_truth_fn'):
        # keyword: strict 매칭 (정확한 숫자/코드)
        relevant_ids = [d.metadata['id'] for d in pdf_docs if q['ground_truth_fn'](d)]
        method = 'strict'
    else:
        # semantic: LLM-as-Judge
        relevant_ids = label_with_llm(q['query'], pdf_docs, q['domain'], vectorstore=pdf_vectorstore)
        method = 'LLM'
    
    eval_queries.append({
        'query': q['query'],
        'domain': q['domain'],
        'type': q['type'],
        'relevant_ids': relevant_ids,
    })
    print(f'→ {len(relevant_ids):2d}개 ({method})')

print(f'\n라벨링 완료')
kw_avg = sum(len(q['relevant_ids']) for q in eval_queries if q['type']=='keyword') / max(1, sum(1 for q in eval_queries if q['type']=='keyword'))
sem_avg = sum(len(q['relevant_ids']) for q in eval_queries if q['type']=='semantic') / max(1, sum(1 for q in eval_queries if q['type']=='semantic'))
print(f'  keyword  평균 관련 청크: {kw_avg:.1f}개 (strict → 소수)')
print(f'  semantic 평균 관련 청크: {sem_avg:.1f}개 (LLM → 다수)')
print(f'\n→ keyword는 정답이 1~2개뿐이라 정확한 매칭이 중요 (BM25 유리)')
print(f'→ semantic은 정답이 여러 개라 의미 파악이 중요 (Vector 유리)')

Ground Truth 라벨링 진행 중...
  [ 1/12] [keyword ] 삼성전기 목표주가 200,000원 투자의견 BUY... →  2개 (strict)
  [ 2/12] [keyword ] 삼성전기 EPS 8,999원 PER 17.4배 2024F... →  1개 (strict)
  [ 3/12] [semantic] 이 회사 주식을 지금 사도 되는지, 앞으로 실적이 좋아질 근거... → 12개 (LLM)
  [ 4/12] [semantic] 전자부품 각 사업 부문의 실적이 어떻게 변하고 있는지... →  2개 (LLM)
  [ 5/12] [keyword ] 제26조 해고의 예고 30일 전 통상임금 30일분... →  1개 (strict)
  [ 6/12] [keyword ] 제37조 지연이자 연 100분의 40 미지급 임금... →  1개 (strict)
  [ 7/12] [semantic] 직장에서 부당하게 쫓겨났을 때 도움받을 수 있는 곳과 절차... → 11개 (LLM)
  [ 8/12] [semantic] 회사가 월급을 제때 안 줄 때 받을 수 있는 벌칙성 이자... →  3개 (LLM)
  [ 9/12] [keyword ] BLEU score 28.4 EN-DE WMT 2014 translation... →  0개 (strict)
  [10/12] [keyword ] d_model 512 d_ff 2048 h 8 heads Transformer... →  2개 (strict)
  [11/12] [semantic] 순서 정보가 없는 구조에서 단어 위치를 알려주기 위해 추가하는 신호... →  7개 (LLM)
  [12/12] [semantic] 순환 구조의 순차 처리 병목을 제거하고 모든 위치를 동시에 처리하는 방법... → 19개 (LLM)

라벨링 완료
  keyword  평균 관련 청크: 1.2개 (strict → 소수)
  semantic 평균 관련 청크: 9.0개 (LLM → 다수)

→ keyword는 정답이 1~2개뿐이라 정확한 

In [21]:
# --- 8-5. 라벨 검증 (Spot Check) ---
print('라벨 검증 — 각 쿼리별 관련 청크 샘플:')
print('=' * 70)

for q in eval_queries:
    print(f'\n쿼리: {q["query"]}')
    print(f'  도메인: {q["domain"]} | 유형: {q["type"]} | 관련 청크: {len(q["relevant_ids"])}개')
    if q['relevant_ids']:
        for rid in q['relevant_ids'][:2]:
            chunk = next((d for d in pdf_docs if d.metadata['id'] == rid), None)
            if chunk:
                preview = chunk.page_content[:100].replace('\n', ' ')
                print(f'    ✓ {rid}: {preview}...')
    else:
        print(f'    ⚠️ 관련 청크 없음 — 쿼리 수정 필요')
    print(f'  {"─" * 60}')

라벨 검증 — 각 쿼리별 관련 청크 샘플:

쿼리: 삼성전기 목표주가 200,000원 투자의견 BUY
  도메인: finance | 유형: keyword | 관련 청크: 2개
    ✓ samsung_electro_chunk_000: Korea | 전기∙전자 | 2024.05.31  삼성전기  (009150)    투자의견  BUY(유지)  목표주가  200,000 원(상향)  현재주가  156,600 원(5/...
    ✓ samsung_electro_chunk_003: 1M  6M  12M  상대기준  2.6  3.5  3.4  절대기준  0.4  7.5  5.3    (원, 십억원)  현재  직전  변동  투자의견  BUY  BUY  -  목표...
  ────────────────────────────────────────────────────────────

쿼리: 삼성전기 EPS 8,999원 PER 17.4배 2024F
  도메인: finance | 유형: keyword | 관련 청크: 1개
    ✓ samsung_electro_chunk_003: 1M  6M  12M  상대기준  2.6  3.5  3.4  절대기준  0.4  7.5  5.3    (원, 십억원)  현재  직전  변동  투자의견  BUY  BUY  -  목표...
  ────────────────────────────────────────────────────────────

쿼리: 이 회사 주식을 지금 사도 되는지, 앞으로 실적이 좋아질 근거
  도메인: finance | 유형: semantic | 관련 청크: 12개
    ✓ samsung_electro_chunk_000: Korea | 전기∙전자 | 2024.05.31  삼성전기  (009150)    투자의견  BUY(유지)  목표주가  200,000 원(상향)  현재주가  156,600 원(5/...
    ✓ samsung_electro_chunk_001: 이른 플래그쉽 모델 출시로 인해 고부가제품 중심의 믹스 개선 

In [27]:
# --- 8-6. eval_queries 저장/로드 ---
# LLM-as-Judge 라벨링은 API 비용이 들므로 한 번만 실행하고 결과를 저장합니다.
import json as json_lib

EVAL_QUERIES_PATH = 'eval_queries_cache.json'

def save_eval_queries(queries, path=EVAL_QUERIES_PATH):
    """eval_queries를 JSON으로 저장 (ground_truth_fn 제외)"""
    serializable = [
        {k: v for k, v in q.items() if k != 'ground_truth_fn'}
        for q in queries
    ]
    with open(path, 'w', encoding='utf-8') as f:
        json_lib.dump(serializable, f, ensure_ascii=False, indent=2)
    print(f'저장 완료: {path} ({len(queries)}개 쿼리)')

def load_eval_queries(path=EVAL_QUERIES_PATH):
    """저장된 eval_queries 로드"""
    with open(path, 'r', encoding='utf-8') as f:
        queries = json_lib.load(f)
    print(f'로드 완료: {path} ({len(queries)}개 쿼리)')
    return queries

# 저장
save_eval_queries(eval_queries)


저장 완료: eval_queries_cache.json (12개 쿼리)


In [28]:
# 다음 실행 시 라벨링 스킵하고 바로 로드하려면:
eval_queries = load_eval_queries()

print(len(eval_queries))

로드 완료: eval_queries_cache.json (12개 쿼리)
12


---
## 9. 실전 데이터 Retriever 구축

PDF 청크 데이터로 BM25 · 벡터 · 하이브리드 리트리버를 새로 구축합니다.

> 섹션 4-6의 인라인 문서용 리트리버와 구분하기 위해 `pdf_` 접두어를 사용합니다.

In [30]:
# --- 9-1. BM25 Retriever (PDF) ---
from langchain_community.retrievers import BM25Retriever

K = 10  # 검색 결과 수

pdf_bm25 = BM25Retriever.from_documents(pdf_docs, k=K)
print(f'BM25 인덱스 구축 완료: {len(pdf_docs)} 청크')

# 간단한 검증
test = pdf_bm25.invoke('연차 유급휴가')
print(f'검증 쿼리 결과: {len(test)}개 청크 반환')
print(f'  첫 결과: [{test[0].metadata["source"]}] {test[0].page_content[:80]}...')

BM25 인덱스 구축 완료: 220 청크
검증 쿼리 결과: 10개 청크 반환
  첫 결과: [labor_law] 1. 임금
2. 소정근로시간
3. 제55조에 따른 휴일
4. 제60조에 따른 연차 유급휴가
5. 그 밖에 대통령령으로 정하는 근로조건
법제처  ...


In [23]:
# --- 9-2. Vector Retriever (PDF) ---
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 다국어 임베딩 모델 (한국어 + 영어 동시 지원)
embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

pdf_vectorstore = Chroma.from_documents(
    documents=pdf_docs,
    embedding=embed_model,
    collection_name='pdf_hybrid_search',
)
pdf_vector = pdf_vectorstore.as_retriever(search_kwargs={'k': K})
print(f'벡터 인덱스 구축 완료: {len(pdf_docs)} 청크 ({embed_model.model_name})')

# 검증
test = pdf_vector.invoke('scaled dot-product attention formula')
print(f'검증: {len(test)}개 반환, 첫 결과 source={test[0].metadata["source"]}')

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 70795.29it/s]


벡터 인덱스 구축 완료: 220 청크 (BAAI/bge-m3)
검증: 10개 반환, 첫 결과 source=transformer


In [24]:
# --- 9-3. Hybrid RRF Retriever (PDF) ---
# 섹션 6에서 정의한 PandasHybridRRFRetriever 재사용

pdf_hybrid = PandasHybridRRFRetriever(
    retrievers=[pdf_bm25, pdf_vector],
    weights=[0.5, 0.5],
    k=60,  # RRF k 파라미터
)

# 테스트 쿼리
test = pdf_hybrid.invoke('연차 유급휴가 발생 요건')
print(f'하이브리드 검증: {len(test)}개 반환')
for i, doc in enumerate(test[:3]):
    print(f'  [{i+1}] {doc.metadata["source"]:20s} | {doc.page_content[:60]}...')

하이브리드 검증: 13개 반환
  [1] labor_law            | 제62조(유급휴가의 대체) 사용자는 근로자대표와의 서면 합의에 따라 제60조에 따른 연차 유급휴가일을 갈음하...
  [2] labor_law            | 수를 알려주고, 근로자가 그 사용 시기를 정하여 사용자에게 통보하도록 서면으로 촉구할 것. 다만, 사용자가 ...
  [3] labor_law            | 1., 2017. 11. 28.>
1. 근로자가 업무상의 부상 또는 질병으로 휴업한 기간
2. 임신 중의 여...


---
## 10. 체계적 Recall@K 실험

### 실험 설계

| 변수 | 값 |
|------|----|
| Retriever | BM25, Vector, Hybrid (RRF) |
| K 값 | 1, 3, 5, 10 |
| 도메인 | finance (금융), law (법률), ai (AI) |
| 쿼리 유형 | keyword, semantic, mixed |

총 **3 × 4 = 12** 조합의 Recall@K를 측정하고, 도메인 · 쿼리유형별로 분석합니다.

In [25]:
# --- 10-1. 실전 Recall@K 측정 함수 (PDF용) ---
import time
import numpy as np

def eval_retriever_pdf(
    retriever,
    queries: list,
    k: int = 10,
    retriever_name: str = 'unknown',
) -> list:
    """
    PDF 기반 Recall@K 평가.
    
    Returns: list of per-query result dicts
    """
    results = []
    for q in queries:
        start = time.perf_counter()
        retrieved = retriever.invoke(q['query'])
        latency = time.perf_counter() - start
        
        retrieved_ids = [d.metadata['id'] for d in retrieved[:k]]
        relevant_ids = set(q['relevant_ids'])
        
        if len(relevant_ids) == 0:
            recall = 0.0
        else:
            hits = len(relevant_ids & set(retrieved_ids))
            recall = hits / len(relevant_ids)
        
        # MRR: 첫 번째 관련 문서의 역순위
        mrr = 0.0
        for rank_pos, rid in enumerate(retrieved_ids, 1):
            if rid in relevant_ids:
                mrr = 1.0 / rank_pos
                break

        results.append({
            'retriever': retriever_name,
            'query': q['query'][:50],
            'domain': q['domain'],
            'type': q['type'],
            'k': k,
            'recall': recall,
            'hits': hits if len(relevant_ids) > 0 else 0,
            'total_relevant': len(relevant_ids),
            'mrr': mrr,
            'latency_ms': latency * 1000,
        })
    return results

In [26]:
# --- 10-2. 전체 실험 실행 ---
import pandas as pd

retrievers = {
    'BM25':   pdf_bm25,
    'Vector': pdf_vector,
    'Hybrid': pdf_hybrid,
}

k_values = [1, 3, 5, 10]

all_results = []
print('실험 진행 중...')
for name, ret in retrievers.items():
    for k in k_values:
        res = eval_retriever_pdf(ret, eval_queries, k=k, retriever_name=name)
        all_results.extend(res)
    print(f'  ✓ {name} 완료 ({len(k_values)} K값 × {len(eval_queries)} 쿼리)')

df_results = pd.DataFrame(all_results)
print(f'\n총 실험 결과: {len(df_results)} rows')
print(f'  컬럼: {list(df_results.columns)}')

실험 진행 중...
  ✓ BM25 완료 (4 K값 × 12 쿼리)
  ✓ Vector 완료 (4 K값 × 12 쿼리)
  ✓ Hybrid 완료 (4 K값 × 12 쿼리)

총 실험 결과: 144 rows
  컬럼: ['retriever', 'query', 'domain', 'type', 'k', 'recall', 'hits', 'total_relevant', 'mrr', 'latency_ms']


In [31]:
# --- 10-3. 전체 Recall@K + MRR@K 요약 ---
print('=' * 70)
print('              Recall@K 요약 (Overall Mean)')
print('=' * 70)

pivot_recall = df_results.pivot_table(
    index='retriever', columns='k', values='recall', aggfunc='mean'
).reindex(['BM25', 'Vector', 'Hybrid'])
print(pivot_recall.round(3).to_string())

print()
print('=' * 70)
print('              MRR@K 요약 — 정답이 몇 위에 있는가')
print('=' * 70)

pivot_mrr = df_results.pivot_table(
    index='retriever', columns='k', values='mrr', aggfunc='mean'
).reindex(['BM25', 'Vector', 'Hybrid'])
print(pivot_mrr.round(3).to_string())

print()
# 쿼리 유형별 Recall@5 + MRR@5
print('=' * 70)
print('          쿼리 유형별 Recall@5 / MRR@5')
print('=' * 70)
df_k5 = df_results[df_results['k'] == 5]
for metric in ['recall', 'mrr']:
    label = 'Recall@5' if metric == 'recall' else 'MRR@5'
    pivot = df_k5.pivot_table(
        index='retriever', columns='type', values=metric, aggfunc='mean'
    ).reindex(['BM25', 'Vector', 'Hybrid'])
    if 'keyword' in pivot.columns:
        pivot = pivot[['keyword', 'semantic']]
    print(f'\n  [{label}]')
    print(pivot.round(3).to_string())

# 최고 성능
best = df_results.groupby(['retriever', 'k'])['recall'].mean()
best_idx = best.idxmax()
print(f'\n최고 Recall: {best_idx[0]} @ K={best_idx[1]} → {best.max():.3f}')

best_mrr = df_results.groupby(['retriever', 'k'])['mrr'].mean()
best_mrr_idx = best_mrr.idxmax()
print(f'최고 MRR:    {best_mrr_idx[0]} @ K={best_mrr_idx[1]} → {best_mrr.max():.3f}')

              Recall@K 요약 (Overall Mean)
k             1      3      5      10
retriever                            
BM25       0.215  0.299  0.299  0.299
Vector     0.267  0.393  0.454  0.529
Hybrid     0.239  0.419  0.529  0.570

              MRR@K 요약 — 정답이 몇 위에 있는가
k             1      3      5      10
retriever                            
BM25       0.333  0.375  0.375  0.375
Vector     0.667  0.722  0.722  0.722
Hybrid     0.583  0.708  0.708  0.708

          쿼리 유형별 Recall@5 / MRR@5

  [Recall@5]
type       keyword  semantic
retriever                   
BM25         0.583     0.014
Vector       0.583     0.324
Hybrid       0.583     0.474

  [MRR@5]
type       keyword  semantic
retriever                   
BM25         0.583     0.167
Vector       0.444     1.000
Hybrid       0.500     0.917

최고 Recall: Hybrid @ K=10 → 0.570
최고 MRR:    Vector @ K=3 → 0.722


In [32]:
# --- 10-4. 도메인별 분석 ---
print('=' * 70)
print('                도메인별 Recall@10 비교')
print('=' * 70)

df_k10 = df_results[df_results['k'] == 10]

pivot_domain = df_k10.pivot_table(
    index='retriever',
    columns='domain',
    values='recall',
    aggfunc='mean',
).reindex(['BM25', 'Vector', 'Hybrid'])

pivot_domain['overall'] = df_k10.groupby('retriever')['recall'].mean()
print(pivot_domain.round(3).to_string())
print()

# 쿼리 유형별 분석
print('--- 쿼리 유형별 Recall@10 ---')
pivot_type = df_k10.pivot_table(
    index='retriever',
    columns='type',
    values='recall',
    aggfunc='mean',
).reindex(['BM25', 'Vector', 'Hybrid'])
print(pivot_type.round(3).to_string())
print()

# 핵심 인사이트
print('--- 핵심 인사이트 ---')
for domain in ['law', 'ai']:
    dom_data = pivot_domain[domain]
    best_ret = dom_data.idxmax()
    print(f'  {domain:3s} 도메인 최고: {best_ret} ({dom_data.max():.3f})')

for qtype in ['keyword', 'semantic', 'mixed']:
    if qtype in pivot_type.columns:
        type_data = pivot_type[qtype]
        best_ret = type_data.idxmax()
        print(f'  {qtype:8s} 쿼리 최고: {best_ret} ({type_data.max():.3f})')

                도메인별 Recall@10 비교
domain        ai  finance    law  overall
retriever                                
BM25       0.125    0.271  0.500    0.299
Vector     0.370    0.458  0.758    0.529
Hybrid     0.370    0.583  0.758    0.570

--- 쿼리 유형별 Recall@10 ---
type       keyword  semantic
retriever                   
BM25         0.583     0.014
Vector       0.583     0.474
Hybrid       0.667     0.474

--- 핵심 인사이트 ---
  law 도메인 최고: Vector (0.758)
  ai  도메인 최고: Vector (0.370)
  keyword  쿼리 최고: Hybrid (0.667)
  semantic 쿼리 최고: Vector (0.474)


In [33]:
# --- 10-5. Latency 분석 ---
print('=' * 70)
print('             평균 Latency (ms) by Retriever × K')
print('=' * 70)

pivot_latency = df_results.pivot_table(
    index='retriever',
    columns='k',
    values='latency_ms',
    aggfunc='mean',
).reindex(['BM25', 'Vector', 'Hybrid'])

print(pivot_latency.round(1).to_string())
print()
print('→ Hybrid는 BM25 + Vector를 순차 실행하므로 latency가 더 큼.')
print('  프로덕션에서는 병렬 실행으로 개선 가능.')

             평균 Latency (ms) by Retriever × K
k            1     3     5     10
retriever                        
BM25        0.2   0.2   0.1   0.1
Vector     65.0  63.8  64.6  63.9
Hybrid     63.7  65.8  64.4  64.4

→ Hybrid는 BM25 + Vector를 순차 실행하므로 latency가 더 큼.
  프로덕션에서는 병렬 실행으로 개선 가능.


---
## 11. RRF 가중치 최적화 — Grid Search

Hybrid 검색의 핵심 하이퍼파라미터:

| 파라미터 | 역할 | 탐색 범위 |
|----------|------|-----------|
| `bm25_weight` | BM25 기여도 | 0.1 ~ 0.9 |
| `vector_weight` | 벡터 검색 기여도 | 1 - bm25_weight |
| `rrf_k` | RRF 스무딩 상수 | 10, 30, 60, 100 |

Grid search로 **Recall@5가 가장 높은 조합**을 찾습니다.

In [34]:
# --- 11-1. Grid Search ---
from itertools import product

bm25_weights = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
rrf_k_values = [10, 30, 60, 100]

grid_results = []
total = len(bm25_weights) * len(rrf_k_values)
print(f'Grid Search: {total}개 조합 탐색...')

for idx, (bw, rk) in enumerate(product(bm25_weights, rrf_k_values), 1):
    vw = round(1.0 - bw, 1)
    hybrid = PandasHybridRRFRetriever(
        retrievers=[pdf_bm25, pdf_vector],
        weights=[bw, vw],
        k=rk,
    )
    res = eval_retriever_pdf(hybrid, eval_queries, k=5, retriever_name=f'w={bw}/k={rk}')
    avg_recall = np.mean([r['recall'] for r in res])
    avg_latency = np.mean([r['latency_ms'] for r in res])
    
    # 도메인별
    finance_recall = np.mean([r['recall'] for r in res if r['domain'] == 'finance'])
    law_recall = np.mean([r['recall'] for r in res if r['domain'] == 'law'])
    ai_recall = np.mean([r['recall'] for r in res if r['domain'] == 'ai'])
    
    grid_results.append({
        'bm25_weight': bw,
        'vector_weight': vw,
        'rrf_k': rk,
        'recall_5': avg_recall,
        'recall_finance': finance_recall,
        'recall_law': law_recall,
        'recall_ai': ai_recall,
        'latency_ms': avg_latency,
    })
    if idx % 7 == 0 or idx == total:
        print(f'  {idx}/{total} 완료...')

df_grid = pd.DataFrame(grid_results).sort_values('recall_5', ascending=False)
print(f'\n탐색 완료: {len(df_grid)}개 조합')

Grid Search: 28개 조합 탐색...
  7/28 완료...
  14/28 완료...
  21/28 완료...
  28/28 완료...

탐색 완료: 28개 조합


In [35]:
# --- 11-2. 최적 조합 분석 ---
print('=' * 70)
print('          Top-5 가중치 조합 (Recall@5 기준)')
print('=' * 70)
print(df_grid.head(10).to_string(index=False))
print()

best = df_grid.iloc[0]
print(f'\n최적 조합:')
print(f'  BM25 가중치:   {best["bm25_weight"]}')
print(f'  Vector 가중치: {best["vector_weight"]}')
print(f'  RRF k:         {best["rrf_k"]}')
print(f'  Recall@5:     {best["recall_5"]:.4f}')
print(f'  금융 도메인:   {best["recall_finance"]:.4f}')
print(f'  법률 도메인:   {best["recall_law"]:.4f}')
print(f'  AI 도메인:     {best["recall_ai"]:.4f}')

# 최적 파라미터로 최종 Retriever 구축
pdf_hybrid_tuned = PandasHybridRRFRetriever(
    retrievers=[pdf_bm25, pdf_vector],
    weights=[best['bm25_weight'], best['vector_weight']],
    k=int(best['rrf_k']),
)
print('\n최적 Hybrid Retriever 구축 완료 (pdf_hybrid_tuned)')

          Top-5 가중치 조합 (Recall@5 기준)
 bm25_weight  vector_weight  rrf_k  recall_5  recall_finance  recall_law  recall_ai  latency_ms
         0.2            0.8     10  0.528737        0.458333    0.757576   0.370301   67.082538
         0.4            0.6     60  0.528737        0.458333    0.757576   0.370301   64.650781
         0.6            0.4    100  0.528737        0.458333    0.757576   0.370301   64.388781
         0.6            0.4     60  0.528737        0.458333    0.757576   0.370301   64.948899
         0.6            0.4     30  0.528737        0.458333    0.757576   0.370301   64.801938
         0.5            0.5    100  0.528737        0.458333    0.757576   0.370301   67.118927
         0.2            0.8     30  0.528737        0.458333    0.757576   0.370301   64.972851
         0.5            0.5     30  0.528737        0.458333    0.757576   0.370301   64.719365
         0.5            0.5     10  0.528737        0.458333    0.757576   0.370301   65.944865
   

---
## 12. 청크 크기가 Recall@K에 미치는 영향

청크 크기(chunk size)는 RAG 파이프라인 성능에 큰 영향을 미칩니다.
너무 작으면 컨텍스트가 부족하고, 너무 크면 노이즈가 증가합니다.

3가지 청크 크기로 실험합니다: **300, 500, 800**

In [37]:
# --- 12-1. 청크 크기별 Recall@5 실험 ---
chunk_sizes = [300, 500, 800]
chunk_exp_results = []

for cs in chunk_sizes:
    print(f'\n청크 크기 = {cs} 실험 중...')
    
    # 새 청킹
    sp = RecursiveCharacterTextSplitter(
        chunk_size=cs,
        chunk_overlap=int(cs * 0.2),
        separators=['\n\n', '\n', '. '],
    )
    cs_docs = []
    for key, text in raw_texts.items():
        if 'samsung' in key:
            domain = 'finance'
        elif 'labor' in key:
            domain = 'law'
        else:
            domain = 'ai'
        chunks = sp.split_text(text)
        for i, chunk in enumerate(chunks):
            cs_docs.append(Document(
                page_content=chunk,
                metadata={'id': f'{key}_cs{cs}_{i:03d}', 'source': key, 'domain': domain}
            ))
    
    print(f'  청크 수: {len(cs_docs)}')
    
    # 리트리버 구축
    cs_bm25 = BM25Retriever.from_documents(cs_docs, k=5)
    cs_vs = Chroma.from_documents(cs_docs, embed_model, collection_name=f'chunk_{cs}')
    cs_vec = cs_vs.as_retriever(search_kwargs={'k': 5})
    cs_hybrid = PandasHybridRRFRetriever(
        retrievers=[cs_bm25, cs_vec],
        weights=[best['bm25_weight'], best['vector_weight']],
        k=int(best['rrf_k']),
    )
    
    # 쿼리의 relevant_ids를 새 청크 기준으로 재계산
    # eval_queries_raw의 ground_truth_fn(keyword) 또는 LLM 재라벨링(semantic)
    cs_queries = []
    for q_raw, q_eval in zip(eval_queries_raw, eval_queries):
        new_q = q_eval.copy()
        if q_raw['type'] == 'keyword' and q_raw.get('ground_truth_fn'):
            # strict 매칭: 새 청크에서 재계산
            new_q['relevant_ids'] = [d.metadata['id'] for d in cs_docs if q_raw['ground_truth_fn'](d)]
        else:
            # semantic: LLM 재라벨링 (비용 절감을 위해 도메인 필터 + 샘플링)
            new_q['relevant_ids'] = label_with_llm(q_raw['query'], cs_docs, q_raw['domain'], vectorstore=cs_vs)
        cs_queries.append(new_q)
    
    # 평가
    for name, ret in [('BM25', cs_bm25), ('Vector', cs_vec), ('Hybrid', cs_hybrid)]:
        res = eval_retriever_pdf(ret, cs_queries, k=5, retriever_name=name)
        avg_recall = np.mean([r['recall'] for r in res])
        chunk_exp_results.append({
            'chunk_size': cs,
            'retriever': name,
            'recall_5': avg_recall,
            'num_chunks': len(cs_docs),
        })
    print(f'  ✓ 완료')

df_chunk = pd.DataFrame(chunk_exp_results)
print('\n--- 청크 크기별 Recall@5 ---')
print(df_chunk.pivot_table(
    index='retriever', columns='chunk_size', values='recall_5'
).reindex(['BM25', 'Vector', 'Hybrid']).round(3).to_string())


청크 크기 = 300 실험 중...
  청크 수: 348


KeyboardInterrupt: 

In [ ]:
# --- 12-2. 청크 실험 결과 저장/로드 ---
# 청크별 LLM 라벨링에 API 호출이 많으므로 결과를 캐싱합니다.

CHUNK_CACHE_PATH = 'chunk_experiment_cache.csv'

# 저장
df_chunk.to_csv(CHUNK_CACHE_PATH, index=False)
print(f'저장 완료: {CHUNK_CACHE_PATH} ({len(df_chunk)}행)')
display(df_chunk)

In [ ]:
# 다음 실행 시 라벨링 스킵하고 바로 로드:
df_chunk = pd.read_csv(CHUNK_CACHE_PATH)

---
## 13. 결과 시각화

전체 실험 결과를 전문가 수준 차트로 정리합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # macOS 한글
matplotlib.rcParams['axes.unicode_minus'] = False

COLORS = {'BM25': '#4ECDC4', 'Vector': '#FF6B6B', 'Hybrid': '#45B7D1'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EP01. Hybrid Search — Recall@K 실험 결과', fontsize=16, fontweight='bold')

# (1) Recall@K 곡선
ax = axes[0, 0]
for name in ['BM25', 'Vector', 'Hybrid']:
    subset = df_results[df_results['retriever'] == name]
    means = subset.groupby('k')['recall'].mean()
    ax.plot(means.index, means.values, 'o-', label=name, color=COLORS[name], linewidth=2, markersize=6)
ax.set_xlabel('K')
ax.set_ylabel('Recall@K')
ax.set_title('(a) Recall@K 곡선')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xticks(k_values)

# (2) 도메인별 Recall@10 비교
ax = axes[0, 1]
domain_data = df_k10.groupby(['retriever', 'domain'])['recall'].mean().unstack()
domain_data = domain_data.reindex(['BM25', 'Vector', 'Hybrid'])
x = np.arange(len(domain_data))
w = 0.3
ax.bar(x - w/2, domain_data['law'], w, label='법률 (한국어)', color='#FFB347')
ax.bar(x + w/2, domain_data['ai'], w, label='AI (영어)', color='#87CEEB')
ax.set_xticks(x)
ax.set_xticklabels(domain_data.index)
ax.set_ylabel('Recall@10')
ax.set_title('(b) 도메인별 Recall@10')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# (3) RRF 가중치 히트맵 (rrf_k=최적값 고정)
ax = axes[1, 0]
best_rrf_k = int(best['rrf_k'])
heatmap_data = df_grid[df_grid['rrf_k'] == best_rrf_k].pivot_table(
    index='bm25_weight', values='recall_10'
)
ax.plot(heatmap_data.index, heatmap_data['recall_10'], 's-', color='#45B7D1', linewidth=2, markersize=8)
ax.fill_between(heatmap_data.index, heatmap_data['recall_10'], alpha=0.2, color='#45B7D1')
ax.set_xlabel('BM25 가중치 (Vector = 1 - BM25)')
ax.set_ylabel('Recall@10')
ax.set_title(f'(c) BM25 가중치 vs Recall@10 (RRF k={best_rrf_k})')
ax.grid(True, alpha=0.3)
ax.axvline(best['bm25_weight'], color='red', linestyle='--', alpha=0.7, label=f'최적: {best["bm25_weight"]}')
ax.legend()

# (4) 청크 크기 영향
ax = axes[1, 1]
for name in ['BM25', 'Vector', 'Hybrid']:
    subset = df_chunk[df_chunk['retriever'] == name]
    ax.plot(subset['chunk_size'], subset['recall_10'], 'o-', label=name, color=COLORS[name], linewidth=2)
ax.set_xlabel('청크 크기 (chars)')
ax.set_ylabel('Recall@10')
ax.set_title('(d) 청크 크기 vs Recall@10')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('recall_experiment_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ recall_experiment_results.png 저장 완료')

In [ ]:
# --- 13-2. 쿼리별 상세 히트맵 ---
fig, ax = plt.subplots(figsize=(12, 6))

# Recall@10 쿼리별 × 리트리버별 히트맵
heatmap_df = df_k10.pivot_table(
    index='query',
    columns='retriever',
    values='recall',
).reindex(columns=['BM25', 'Vector', 'Hybrid'])

im = ax.imshow(heatmap_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(heatmap_df.columns)))
ax.set_xticklabels(heatmap_df.columns, fontsize=12)
ax.set_yticks(range(len(heatmap_df.index)))
ax.set_yticklabels(heatmap_df.index, fontsize=9)
ax.set_title('쿼리별 Recall@10 히트맵', fontsize=14, fontweight='bold')

# 셀 값 표시
for i in range(len(heatmap_df.index)):
    for j in range(len(heatmap_df.columns)):
        val = heatmap_df.values[i, j]
        color = 'white' if val < 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', color=color, fontsize=10)

plt.colorbar(im, ax=ax, label='Recall@10')
plt.tight_layout()
plt.savefig('recall_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ recall_heatmap.png 저장 완료')

---
## 14. 실패 분석 (Error Analysis)

Recall이 낮은 쿼리를 분석하여 **왜** 검색이 실패했는지 파악합니다.
이 분석이 실무에서 검색 파이프라인을 개선하는 핵심 단서가 됩니다.

In [ ]:
# --- 14-1. 실패 쿼리 분석 ---
# Hybrid Recall@10 기준 실패 케이스 (recall < 1.0)
df_hybrid_k10 = df_k10[df_k10['retriever'] == 'Hybrid'].copy()
failures = df_hybrid_k10[df_hybrid_k10['recall'] < 1.0].sort_values('recall')

print(f'Hybrid Recall@10 < 1.0 인 쿼리: {len(failures)}개 / {len(df_hybrid_k10)}개')
print('=' * 80)

for _, row in failures.iterrows():
    print(f'\n쿼리: {row["query"]}')
    print(f'  도메인: {row["domain"]} | 유형: {row["type"]}')
    print(f'  Recall: {row["recall"]:.2f} (hits={row["hits"]}/{row["total_relevant"]})')
    
    # 실제 검색 결과 확인
    q_full = [q for q in eval_queries if q['query'][:50] == row['query']][0]
    retrieved = pdf_hybrid_tuned.invoke(q_full['query'])[:10]
    retrieved_ids = set(d.metadata['id'] for d in retrieved)
    missed = set(q_full['relevant_ids']) - retrieved_ids
    
    if missed:
        print(f'  놓친 청크 ({len(missed)}개):')
        for mid in list(missed)[:3]:
            chunk = [d for d in pdf_docs if d.metadata['id'] == mid]
            if chunk:
                print(f'    → {mid}: {chunk[0].page_content[:80]}...')

print('\n--- 실패 원인 패턴 ---')
print('1. 키워드 불일치: 쿼리 표현과 청크 내 표현이 다름 (동의어, 축약어)')
print('2. 의미적 거리: 벡터 검색에서 관련 문서가 다른 도메인 문서보다 멀리 위치')
print('3. 청크 분할 이슈: 관련 정보가 2개 청크에 걸쳐 분리됨')

---
## 15. Langfuse 통합

실험 결과를 Langfuse에 기록하여 대시보드에서 추적합니다.

`.env` 파일에 `LANGFUSE_PUBLIC_KEY`와 `LANGFUSE_SECRET_KEY`가 설정되어 있으면 자동으로 활성화됩니다.

In [ ]:
# --- 15-1. Langfuse 실험 기록 ---
LANGFUSE_PUBLIC_KEY = os.getenv('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.getenv('LANGFUSE_SECRET_KEY', '')
USE_LANGFUSE = bool(LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY)

if USE_LANGFUSE:
    from langfuse import Langfuse
    
    langfuse = Langfuse()
    
    # 실험 결과를 Langfuse에 Score로 기록
    trace = langfuse.trace(name='ep01-hybrid-search-experiment')
    
    for _, row in df_grid.head(5).iterrows():
        trace.score(
            name='recall_10',
            value=row['recall_10'],
            comment=f'BM25={row["bm25_weight"]}, Vec={row["vector_weight"]}, k={row["rrf_k"]}',
        )
    
    # 최적 파라미터 기록
    trace.score(
        name='best_recall_10',
        value=best['recall_10'],
        comment=f'BEST: BM25={best["bm25_weight"]}, Vec={best["vector_weight"]}, k={best["rrf_k"]}',
    )
    
    langfuse.flush()
    print('Langfuse에 실험 결과 기록 완료')
    print(f'  Trace URL: {trace.get_trace_url()}')
else:
    print('Langfuse 미설정 — 스킵')
    print('활성화: .env에 LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY 추가')

---
## 16. Exercise

아래 과제를 완료하면 실전 Hybrid Search 파이프라인을 완전히 이해한 것입니다.

### Exercise 1: 새로운 PDF 문서 추가 실험

**목표:** 자신만의 PDF를 `data/` 폴더에 추가하고 Recall@K가 어떻게 변하는지 분석

1. PDF 1개를 `data/` 폴더에 추가
2. 해당 문서에 대한 쿼리 3개 + ground truth 정의
3. 전체 파이프라인 재실행 후 도메인별 Recall@10 비교
4. **분석 포인트:** 새 문서가 기존 문서의 검색 성능에 영향을 주는가?

In [ ]:
# Exercise 1 풀이 공간
# TODO: 새 PDF 로드 및 쿼리 정의

# new_text = load_pdf(os.path.join(DATA_DIR, 'your_document.pdf'))
# new_queries = [
#     {
#         'query': '...',
#         'relevant_fn': make_relevance_fn(['키워드1', '키워드2']),
#         'domain': 'your_domain', 'type': 'keyword',
#     },
# ]


### Exercise 2: 청크 전략 최적화 챌린지

**목표:** 청크 크기, 오버랩, 구분자를 조합하여 **Recall@10 최고 기록 갱신**

1. `RecursiveCharacterTextSplitter`의 파라미터 3개(chunk_size, chunk_overlap, separators)를 변경
2. 최소 5개 조합을 실험
3. 결과를 DataFrame으로 정리하고 최적 조합 도출
4. **핵심 질문:** 오버랩 비율(overlap/chunk_size)이 높을수록 항상 좋은가?

In [ ]:
# Exercise 2 풀이 공간
# TODO: 청크 전략 최적화 실험

# chunk_configs = [
#     {'size': 300, 'overlap': 30, 'sep': ['\n\n', '\n', '. ', ' ']},
#     {'size': 300, 'overlap': 100, 'sep': ['\n\n', '\n', '. ', ' ']},
#     {'size': 500, 'overlap': 50, 'sep': ['\n\n', '\n', '. ', ' ']},
#     ...
# ]


### Exercise 3: 쿼리 확장(Query Expansion)으로 Recall 개선

**목표:** LLM으로 쿼리를 확장하여 BM25 키워드 매칭률을 높인다

1. Claude/GPT에게 원본 쿼리의 동의어/관련어를 생성하도록 프롬프트 작성
2. 확장된 쿼리로 BM25 검색 → Recall@10 측정
3. 원본 vs 확장 쿼리의 Recall 비교
4. **핵심 질문:** 쿼리 확장이 벡터 검색에도 효과가 있는가?

In [ ]:
# Exercise 3 풀이 공간
# TODO: Query Expansion 구현

# from langchain_openai import ChatOpenAI
# llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
#
# def expand_query(query: str) -> str:
#     prompt = f'''원본 쿼리: {query}
#     이 쿼리와 관련된 동의어, 관련어를 추가하여 확장된 검색 쿼리를 생성하세요.
#     형식: 원본 쿼리 + 관련 키워드들 (한 줄)'''
#     return llm.invoke(prompt).content


---
## 마무리

### 이 노트북에서 배운 것

| 개념 | 핵심 포인트 |
|------|------------|
| BM25 | 키워드 기반, 정확한 용어 매칭에 강점 |
| 벡터 검색 | 의미적 유사성 기반, 동의어/다국어에 강점 |
| Hybrid (RRF) | 두 방식 결합, 모든 쿼리 유형에서 가장 안정적 |
| 가중치 튜닝 | Grid Search로 도메인 특화 최적 가중치 탐색 |
| 청크 크기 | 너무 작으면 컨텍스트 부족, 너무 크면 노이즈 증가 |
| 실패 분석 | Recall이 낮은 쿼리의 원인 파악 → 파이프라인 개선 핵심 |

### 다음 에피소드

**EP02. Re-ranking 적용기** — 검색 결과 100개 중 진짜 필요한 5개만 고르는 기법을 다룹니다.